# Brazil E-Commerce Analysis


## 1. Project Overview


### Project Scope

This portfolio project examines Olist's historical Brazilian e-commerce transactions from purchase through payment, delivery, and review. It connects nine CSV datasets at the order, item, customer, product, seller, payment, review, and geographic levels. The analysis covers platform growth, category and seller structure, spending and repeat purchases, regional demand, delivery, and customer ratings.

The unit of analysis changes by question: an order can have multiple items, payments, and reviews, while one customer can place multiple orders. Each chart states the relevant population or denominator so its result can be read alongside the others.


### Project Motivation

Marketplace performance is not captured by a single sales total. A useful view links commercial activity to who buys, what they buy, where orders go, how reliably they arrive, and how customers rate the experience. This notebook builds that view from reproducible joins, explicit metrics, and comparisons across compatible periods and customer cohorts.

The result is a historical Olist operating profile that can inform category monitoring, customer retention tests, geographic prioritization, and delivery-service review.


## 2. Business Objectives

The project evaluates six questions about Olist's observed marketplace activity. Item-price GMV and delivered orders provide the common commercial base; other measures use the eligible records stated in their sections.

| Business question | Main measures | Sections |
|---|---|---|
| How did marketplace activity change? | Delivered orders, item-price GMV, average order value, monthly trend, and an exploratory three-month forecast | 5 |
| Which categories and sellers shaped the product mix? | Category GMV share, Pareto concentration, comparable-month category GMV, prices, charged freight, seller counts, and HHI | 5.3, 6 |
| How did customers pay, spend, and purchase again? | Payment mix, active-month and active-year spend, purchase frequency, RFM groups, value tiers, and fixed-window cohorts | 5.4, 7 |
| Where was activity concentrated? | Customer-state GMV and orders, seller coverage of customer states, and state-level concentration | 8 |
| How did delivery perform? | Delivery duration, estimated-date timeliness, charged freight, and order status | 9 |
| How did customers rate the experience? | Review-score distribution, delivery-timing comparisons, and state review means | 10 |

The nine files support an Olist-level analysis. National market share, brand rankings, household purchasing power, and seller profit require data that are not included here.


## 3. Dataset Overview


### 3.1 Data Sources

The analysis uses nine CSV files from the Olist Brazilian E-Commerce dataset. Together, they describe the order lifecycle, customer and seller locations, products, payments, reviews, and geographic reference data.

| Source File                             | Grain                                    | Main Purpose                                                      |
| --------------------------------------- | ---------------------------------------- | ----------------------------------------------------------------- |
| `olist_orders_dataset.csv`              | One row per order                        | Order status and order lifecycle timestamps.                      |
| `olist_order_items_dataset.csv`         | One row per item within an order         | Item price, freight charge, product, and seller information.      |
| `olist_order_payments_dataset.csv`      | One row per payment record               | Payment method, installment count, and payment value.             |
| `olist_order_reviews_dataset.csv`       | One row per customer review              | Customer rating, review text, and response timestamps.            |
| `olist_customers_dataset.csv`           | One row per customer-order record        | Customer identity and location attributes.                        |
| `olist_products_dataset.csv`            | One row per product                      | Product category and physical product attributes.                 |
| `olist_sellers_dataset.csv`             | One row per seller                       | Seller identity and location attributes.                          |
| `olist_geolocation_dataset.csv`         | One row per geographic coordinate record | Postal-code, city, state, latitude, and longitude reference data. |
| `product_category_name_translation.csv` | One row per product category translation | Portuguese-to-English category mapping.                           |


### 3.2 Data Model and Relationships

The `olist_orders_dataset.csv` file is the central transaction table. It connects each order to the customer, item, payment, and review records needed for marketplace analysis.

| From File                          | Join Key                   | To File                                 | Relationship                                                       |
| ---------------------------------- | -------------------------- | --------------------------------------- | ------------------------------------------------------------------ |
| `olist_orders_dataset.csv`         | `customer_id`              | `olist_customers_dataset.csv`           | Each order links to one customer-order record.                     |
| `olist_order_items_dataset.csv`    | `order_id`                 | `olist_orders_dataset.csv`              | An order can contain multiple item records.                        |
| `olist_order_payments_dataset.csv` | `order_id`                 | `olist_orders_dataset.csv`              | An order can have one or more payment records.                     |
| `olist_order_reviews_dataset.csv`  | `order_id`                 | `olist_orders_dataset.csv`              | An order can be linked to one or more review records.              |
| `olist_order_items_dataset.csv`    | `product_id`               | `olist_products_dataset.csv`            | Each item links to one product.                                    |
| `olist_order_items_dataset.csv`    | `seller_id`                | `olist_sellers_dataset.csv`             | Each item links to one seller.                                     |
| `olist_products_dataset.csv`       | `product_category_name`    | `product_category_name_translation.csv` | Product categories can be translated from Portuguese to English.   |
| `olist_customers_dataset.csv`      | `customer_zip_code_prefix` | `olist_geolocation_dataset.csv`         | Customer locations can be enriched with geographic reference data. |
| `olist_sellers_dataset.csv`        | `seller_zip_code_prefix`   | `olist_geolocation_dataset.csv`         | Seller locations can be enriched with geographic reference data.   |

The geolocation file may contain multiple coordinate records for the same postal-code prefix. Geographic analysis should therefore aggregate this file to one representative location per postal-code prefix before joining it to customer or seller records.


#### Column-Level Relationship Map

Each card is one source CSV and lists its original columns. Bold blue columns are join fields. Colored links group the join types; the exact field pairs and relationship cardinalities are listed in the table above. The raw geolocation file is not unique by postal-code prefix, so aggregate it to one row per prefix before using either location link.

In [1]:
# Visualize source columns and verified join fields using CSV headers only.
from pathlib import Path
import pandas as pd
import plotly.graph_objects as go

schema_dir = Path("source_data")
schema_files = {
    "Geolocation": "olist_geolocation_dataset.csv",
    "Customers": "olist_customers_dataset.csv",
    "Orders": "olist_orders_dataset.csv",
    "Payments": "olist_order_payments_dataset.csv",
    "Reviews": "olist_order_reviews_dataset.csv",
    "Items": "olist_order_items_dataset.csv",
    "Translation": "product_category_name_translation.csv",
    "Products": "olist_products_dataset.csv",
    "Sellers": "olist_sellers_dataset.csv",
}
schema_columns = {
    name: pd.read_csv(schema_dir / filename, nrows=0).columns.tolist()
    for name, filename in schema_files.items()
}
schema_links = [
    ("Orders", "customer_id", "Customers", "customer_id", "Entity key"),
    ("Orders", "order_id", "Payments", "order_id", "Order key"),
    ("Orders", "order_id", "Reviews", "order_id", "Order key"),
    ("Orders", "order_id", "Items", "order_id", "Order key"),
    ("Items", "product_id", "Products", "product_id", "Entity key"),
    ("Items", "seller_id", "Sellers", "seller_id", "Entity key"),
    ("Products", "product_category_name", "Translation", "product_category_name", "Category key"),
    ("Customers", "customer_zip_code_prefix", "Geolocation", "geolocation_zip_code_prefix", "Postal prefix"),
    ("Sellers", "seller_zip_code_prefix", "Geolocation", "geolocation_zip_code_prefix", "Postal prefix"),
]
for left, left_key, right, right_key, _ in schema_links:
    assert left_key in schema_columns[left] and right_key in schema_columns[right]

schema_positions = {
    "Geolocation": (2, 65), "Customers": (26, 95), "Orders": (26, 68),
    "Payments": (51, 95), "Reviews": (51, 69), "Items": (51, 38),
    "Translation": (76, 95), "Products": (76, 77), "Sellers": (76, 38),
}
schema_colors = {
    "Order key": "#2563EB", "Entity key": "#7C3AED",
    "Category key": "#059669", "Postal prefix": "#EA580C",
}
join_columns = {
    name: {
        field for left, left_key, right, right_key, _ in schema_links
        for table, field in [(left, left_key), (right, right_key)]
        if table == name
    }
    for name in schema_files
}
box_width = 22
box_bottom = {
    name: top - (10 + 2.55 * len(schema_columns[name]))
    for name, (_, top) in schema_positions.items()
}
schema_figure = go.Figure()
shown_link_types = set()

def field_y(table, field):
    return schema_positions[table][1] - 10.5 - 2.55 * schema_columns[table].index(field)

for left, left_key, right, right_key, link_type in schema_links:
    lx, _ = schema_positions[left]
    rx, _ = schema_positions[right]
    ly, ry = field_y(left, left_key), field_y(right, right_key)
    if {left, right} == {"Sellers", "Geolocation"}:
        path_x, path_y = [lx, 74, 74, 1, 1, rx], [ly, ly, 4, 4, ry, ry]
    elif lx == rx:
        route_x = 24 if left == "Orders" else 99
        edge_x = lx if route_x < lx else lx + box_width
        path_x, path_y = [edge_x, route_x, route_x, edge_x], [ly, ly, ry, ry]
    elif lx < rx:
        path_x, path_y = [lx + box_width, rx], [ly, ry]
    else:
        path_x, path_y = [lx, rx + box_width], [ly, ry]
    schema_figure.add_trace(go.Scatter(
        x=path_x, y=path_y, mode="lines",
        line={"color": schema_colors[link_type], "width": 3},
        text=f"{left}.{left_key} ↔ {right}.{right_key}",
        hovertemplate="%{text}<extra></extra>",
        name=link_type, showlegend=link_type not in shown_link_types,
    ))
    shown_link_types.add(link_type)

for name, filename in schema_files.items():
    x, top = schema_positions[name]
    bottom = box_bottom[name]
    schema_figure.add_shape(
        type="rect", x0=x, x1=x + box_width, y0=bottom, y1=top,
        fillcolor="#FFFFFF", line={"color": "#94A3B8", "width": 1.5}, layer="above",
    )
    schema_figure.add_shape(
        type="rect", x0=x, x1=x + box_width, y0=top - 8, y1=top,
        fillcolor="#DBEAFE" if name == "Orders" else "#E2E8F0",
        line={"width": 0}, layer="above",
    )
    schema_figure.add_annotation(
        x=x + 1, y=top - 0.6, xanchor="left", yanchor="top",
        text=f"<b>{name}</b><br><span style='font-size:10px'>{filename}</span>",
        showarrow=False, align="left", font={"size": 14, "color": "#0F172A"},
    )
    field_lines = [
        f"<b><span style='color:#1D4ED8'>{field}</span></b>" if field in join_columns[name] else field
        for field in schema_columns[name]
    ]
    schema_figure.add_annotation(
        x=x + 1, y=top - 9.5, xanchor="left", yanchor="top",
        text="<br>".join(field_lines), showarrow=False, align="left",
        font={"size": 12, "color": "#334155"},
    )

schema_figure.update_xaxes(range=[0, 100], visible=False, fixedrange=True)
schema_figure.update_yaxes(range=[0, 100], visible=False, fixedrange=True)
schema_figure.update_layout(
    title="Olist CSV Column Relationships",
    template="plotly_white", width=1650, height=1250,
    legend={"orientation": "h", "x": 0.5, "xanchor": "center", "y": 1.01},
    margin={"l": 25, "r": 25, "t": 110, "b": 25},
    hovermode="closest",
)
schema_figure.show()

### 3.3 Data Dictionary

#### `olist_customers_dataset.csv`

| Field | Data Type | Business Meaning |
|---|---|---|
| `customer_id` | Text | Identifier for the customer record associated with an order. |
| `customer_unique_id` | Text | Persistent identifier for an individual customer across multiple orders. |
| `customer_zip_code_prefix` | Text | Customer postal-code prefix. |
| `customer_city` | Text | Customer city. |
| `customer_state` | Text | Customer state in Brazil. |

**Dataset of customers' information such as id and geolocations.**

#### `olist_geolocation_dataset.csv`

| Field | Data Type | Business Meaning |
|---|---|---|
| `geolocation_zip_code_prefix` | Text | Postal-code prefix used to locate the geographic record. |
| `geolocation_lat` | Decimal | Latitude of the geographic record. |
| `geolocation_lng` | Decimal | Longitude of the geographic record. |
| `geolocation_city` | Text | City associated with the geographic record. |
| `geolocation_state` | Text | Brazilian state associated with the geographic record. |

**Dataset of geolocation positioning.**

#### `olist_order_items_dataset.csv`

| Field | Data Type | Business Meaning |
|---|---|---|
| `order_id` | Text | Identifier of the order containing the item. |
| `order_item_id` | Integer | Sequential number of the item within an order. |
| `product_id` | Text | Identifier of the purchased product. |
| `seller_id` | Text | Identifier of the seller fulfilling the item. |
| `shipping_limit_date` | DateTime | Deadline for the seller to hand the item to the logistics partner. |
| `price` | Decimal | Item price charged to the customer. |
| `freight_value` | Decimal | Freight charge assigned to the order item. |

**Dataset of basic items' selling informations.**

#### `olist_order_payments_dataset.csv`

| Field | Data Type | Business Meaning |
|---|---|---|
| `order_id` | Text | Identifier of the order associated with the payment. |
| `payment_sequential` | Integer | Sequence number for a payment within an order. |
| `payment_type` | Text | Payment method used by the customer. |
| `payment_installments` | Integer | Number of installments selected for the payment. |
| `payment_value` | Decimal | Total monetary value of the payment record. |

**Dataset of orders' paying meathods and amounts.**

#### `olist_order_reviews_dataset.csv`

| Field | Data Type | Business Meaning |
|---|---|---|
| `review_id` | Text | Unique identifier for a customer review. |
| `order_id` | Text | Identifier of the order reviewed by the customer. |
| `review_score` | Integer | Customer satisfaction rating, typically on a five-point scale. |
| `review_comment_title` | Text | Title supplied by the customer for the review. |
| `review_comment_message` | Text | Written comment supplied by the customer. |
| `review_creation_date` | DateTime | Date and time when the customer created the review. |
| `review_answer_timestamp` | DateTime | Date and time when the marketplace responded to the review. |

**Dataset of orders' reviews' rattings, comments and timestamps.**

#### `olist_orders_dataset.csv`

| Field | Data Type | Business Meaning |
|---|---|---|
| `order_id` | Text | Unique identifier for an order. |
| `customer_id` | Text | Identifier of the customer record linked to the order. |
| `order_status` | Text | Current operational status of the order. |
| `order_purchase_timestamp` | DateTime | Date and time when the customer placed the order. |
| `order_approved_at` | DateTime | Date and time when the payment or order approval was completed. |
| `order_delivered_carrier_date` | DateTime | Date and time when the carrier received the order. |
| `order_delivered_customer_date` | DateTime | Date and time when the customer received the order. |
| `order_estimated_delivery_date` | DateTime | Estimated delivery date communicated for the order. |

**Dataset of orders' lifecycle infos.**

#### `olist_products_dataset.csv`

| Field | Data Type | Business Meaning |
|---|---|---|
| `product_id` | Text | Unique identifier for a product. |
| `product_category_name` | Text | Product category label in Portuguese. |
| `product_name_lenght` | Integer | Number of characters in the product name. |
| `product_description_lenght` | Integer | Number of characters in the product description. |
| `product_photos_qty` | Integer | Number of product photographs available. |
| `product_weight_g` | Decimal | Product weight in grams. |
| `product_length_cm` | Decimal | Product length in centimeters. |
| `product_height_cm` | Decimal | Product height in centimeters. |
| `product_width_cm` | Decimal | Product width in centimeters. |

**Dataset of online products infos.**

#### `olist_sellers_dataset.csv`

| Field | Data Type | Business Meaning |
|---|---|---|
| `seller_id` | Text | Unique identifier for a marketplace seller. |
| `seller_zip_code_prefix` | Text | Seller postal-code prefix. |
| `seller_city` | Text | Seller city. |
| `seller_state` | Text | Seller state in Brazil. |

**Dataset of sellers' infos.**

#### `product_category_name_translation.csv`

| Field | Data Type | Business Meaning |
|---|---|---|
| `product_category_name` | Text | Original Portuguese product category label. |
| `product_category_name_english` | Text | English translation of the product category label. |

**Dataset of product categories' English translations.**


## 4. Data Preparation


### 4.1 Environment Setup


In [2]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display
from pathlib import Path

DATA_DIR = Path("source_data")

### 4.2 Data Loading


In [3]:
customers_file = 'olist_customers_dataset.csv'
geolocation_file = 'olist_geolocation_dataset.csv'
items_file = 'olist_order_items_dataset.csv'
payments_file = 'olist_order_payments_dataset.csv'
reviews_file = 'olist_order_reviews_dataset.csv'
orders_file = 'olist_orders_dataset.csv'
products_file = 'olist_products_dataset.csv'
sellers_file = 'olist_sellers_dataset.csv'
translation_file = 'product_category_name_translation.csv'

files = [
    customers_file,
    geolocation_file,
    items_file,
    payments_file,
    reviews_file,
    orders_file,
    products_file,
    sellers_file,
    translation_file]

### 4.3 Data Quality Assessment


In [4]:
quality_results = []

for _ in files:
    df = pd.read_csv(DATA_DIR / _)
    
    total_row = df.shape[0]
    total_cells = df.shape[0]*df.shape[1]
    
    nulls = df.isnull().sum().sum()
    duplicates = df.duplicated().sum()
    
    missing_rate = nulls / total_cells if total_cells > 0 else 0
    duplicate_rate = duplicates / total_row if total_row > 0 else 0

    quality_results.append({
        "Dataset":_,
        "Total Rows": total_row,
        "Columns": df.shape[1],
        "Missing Values": nulls,
        "Missing Rate": missing_rate,
        "Duplicate Rows": duplicates,
        "Duplicate Rate":duplicate_rate
    })
    
quality_summary = pd.DataFrame(quality_results)

quality_summary['Missing Rate'] = quality_summary["Missing Rate"].map(lambda value:f"{value:.2%}")
quality_summary['Duplicate Rate'] = quality_summary['Duplicate Rate'].map(lambda value:f"{value:.2%}")

display(quality_summary)


,Dataset,Total Rows,Columns,Missing Values,Missing Rate,Duplicate Rows,Duplicate Rate
0,olist_customers_dataset.csv,99441,5,0,0.00%,0,0.00%
1,olist_geolocation_dataset.csv,1000163,5,0,0.00%,261831,26.18%
2,olist_order_items_dataset.csv,112650,7,0,0.00%,0,0.00%
3,olist_order_payments_dataset.csv,103886,5,0,0.00%,0,0.00%
4,olist_order_reviews_dataset.csv,99224,7,145903,21.01%,0,0.00%
5,olist_orders_dataset.csv,99441,8,4908,0.62%,0,0.00%
6,olist_products_dataset.csv,32951,9,2448,0.83%,0,0.00%
7,olist_sellers_dataset.csv,3095,4,0,0.00%,0,0.00%
8,product_category_name_translation.csv,71,2,0,0.00%,0,0.00%


Missing review comments and optional order-delivery timestamps do not necessarily indicate data errors. Product description fields may also be absent without invalidating a product ID or its transaction. Exact duplicate geolocation rows are reported separately because the location table contains repeated coordinates at ZIP-prefix grain. Do not interpret a cell-level missing rate as the share of unusable orders, and do not delete all incomplete rows indiscriminately.

### 4.4 Data Cleaning


In [5]:
file_map = {
    "customers_df": "olist_customers_dataset.csv",
    "geolocation_df": "olist_geolocation_dataset.csv",
    "items_df": "olist_order_items_dataset.csv",
    "payments_df": "olist_order_payments_dataset.csv",
    "reviews_df": "olist_order_reviews_dataset.csv",
    "orders_df": "olist_orders_dataset.csv",
    "products_df": "olist_products_dataset.csv",
    "sellers_df": "olist_sellers_dataset.csv",
    "translation_df": "product_category_name_translation.csv",
}

dfs: dict[str, pd.DataFrame] = {}

for df_name, file_name in file_map.items():
    dfs[df_name] = pd.read_csv(DATA_DIR / file_name)

In [6]:
# Validate transaction keys, remove only exact geolocation duplicates, and audit changes.
key_rules = {
    "customers_df": ["customer_id"],
    "items_df": ["order_id", "order_item_id"],
    "payments_df": ["order_id", "payment_sequential"],
    "orders_df": ["order_id"],
    "products_df": ["product_id"],
    "sellers_df": ["seller_id"],
    "translation_df": ["product_category_name"],
}
cleaning_audit = []
for df_name, df in dfs.items():
    before_rows = len(df)
    before_duplicates = int(df.duplicated().sum())
    if df_name in key_rules:
        key_columns = key_rules[df_name]
        if df[key_columns].isna().any().any():
            raise ValueError(f"Required key is missing in {df_name}: {key_columns}")
        if df.duplicated(subset=key_columns).any():
            raise ValueError(f"Required key is not unique in {df_name}: {key_columns}")
    if df_name == "geolocation_df":
        dfs[df_name] = df.drop_duplicates().copy()
    cleaning_audit.append({
        "Dataset": df_name,
        "Input Rows": before_rows,
        "Output Rows": len(dfs[df_name]),
        "Exact Duplicates Removed": before_rows - len(dfs[df_name]),
        "Remaining Exact Duplicates": int(dfs[df_name].duplicated().sum()),
        "Missing Cells Retained": int(dfs[df_name].isna().sum().sum()),
        "Rule": "Remove exact geolocation rows" if df_name == "geolocation_df"
                else "Preserve rows; validate keys" if df_name != "reviews_df"
                else "Preserve raw reviews and optional comments",
    })
cleaning_audit = pd.DataFrame(cleaning_audit)
display(cleaning_audit)
assert cleaning_audit["Remaining Exact Duplicates"].eq(0).all(), (
    "Unexpected exact duplicates require a dataset-specific grain review."
)

,Dataset,Input Rows,Output Rows,Exact Duplicates Removed,Remaining Exact Duplicates,Missing Cells Retained,Rule
0,customers_df,99441,99441,0,0,0,Preserve rows; validate keys
1,geolocation_df,1000163,738332,261831,0,0,Remove exact geolocation rows
2,items_df,112650,112650,0,0,0,Preserve rows; validate keys
3,payments_df,103886,103886,0,0,0,Preserve rows; validate keys
4,reviews_df,99224,99224,0,0,145903,Preserve raw reviews and optional comments
5,orders_df,99441,99441,0,0,4908,Preserve rows; validate keys
6,products_df,32951,32951,0,0,2448,Preserve rows; validate keys
7,sellers_df,3095,3095,0,0,0,Preserve rows; validate keys
8,translation_df,71,71,0,0,0,Preserve rows; validate keys


In [7]:
customers_df = dfs["customers_df"]
geolocation_df = dfs["geolocation_df"]
items_df = dfs["items_df"]
payments_df = dfs["payments_df"]
reviews_df = dfs["reviews_df"]
orders_df = dfs["orders_df"]
products_df = dfs["products_df"]
sellers_df = dfs["sellers_df"]
translation_df = dfs["translation_df"]

### 4.5 Feature Engineering


In [8]:
# Customers Features
customers_df["customer_zip_code_prefix"] = customers_df["customer_zip_code_prefix"].astype("string").str.zfill(5)
customers_df["customer_city"] = customers_df["customer_city"].str.strip().str.lower()


# Geolocation Features
geolocation_df["geolocation_zip_code_prefix"] = geolocation_df["geolocation_zip_code_prefix"].astype("string").str.zfill(5)
geolocation_df["geolocation_lat"] = pd.to_numeric(geolocation_df["geolocation_lat"], errors="coerce")
geolocation_df["geolocation_lng"] = pd.to_numeric(geolocation_df["geolocation_lng"], errors="coerce")


# Order Item Features
items_df["shipping_limit_date"] = pd.to_datetime(items_df["shipping_limit_date"], errors="coerce")
items_df["price"] = pd.to_numeric(items_df["price"], errors="coerce")
items_df["freight_value"] = pd.to_numeric(items_df["freight_value"], errors="coerce")


# Payment Features
payments_df["payment_value"] = pd.to_numeric(payments_df["payment_value"], errors="coerce")
payments_df["payment_installments"] = pd.to_numeric(payments_df["payment_installments"], errors="coerce")


# Review Features
reviews_df["review_creation_date"] = pd.to_datetime(reviews_df["review_creation_date"], errors="coerce")
reviews_df["review_answer_timestamp"] = pd.to_datetime(reviews_df["review_answer_timestamp"], errors="coerce")
reviews_df["review_score"] = pd.to_numeric(reviews_df["review_score"], errors="coerce")


# Order Features
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
for column in date_columns:
    orders_df[column] = pd.to_datetime(orders_df[column], errors="coerce")
orders_df["purchase_year"] = orders_df["order_purchase_timestamp"].dt.year
orders_df["purchase_month"] = orders_df["order_purchase_timestamp"].dt.to_period("M").astype("string")
orders_df["delivery_days"] = (orders_df["order_delivered_customer_date"] - orders_df["order_purchase_timestamp"]).dt.total_seconds() / 86400
orders_df["delivery_delay_days"] = (orders_df["order_delivered_customer_date"] - orders_df["order_estimated_delivery_date"]).dt.total_seconds() / 86400
orders_df["is_late_delivery"] = orders_df["delivery_delay_days"] > 0


# Product Features
products_df["product_category_name"] = products_df["product_category_name"].str.strip().str.lower()
product_numeric_columns = [
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm",
]
for column in product_numeric_columns:
    products_df[column] = pd.to_numeric(products_df[column], errors="coerce")


# Seller Features
sellers_df["seller_zip_code_prefix"] = sellers_df["seller_zip_code_prefix"].astype("string").str.zfill(5)
sellers_df["seller_city"] = sellers_df["seller_city"].str.strip().str.lower()


## 5. Macro Olist Market Analysis

This section establishes the marketplace baseline: delivered-item GMV, order volume, transaction value, time trends, category concentration, and payment behavior.


### 5.1 Market GMV, Average Order Value and Order Volume


In [9]:
# Calculate market-scale KPIs from delivered orders
delivered_order_ids = orders_df.loc[
    orders_df["order_status"].eq("delivered"), "order_id"
].unique()

delivered_items = items_df[items_df["order_id"].isin(delivered_order_ids)]
order_volume = len(delivered_order_ids)
gmv = delivered_items["price"].sum()
average_order_value = gmv / order_volume if order_volume else 0
delivered_customers = orders_df.loc[
    orders_df["order_status"].eq("delivered"), ["customer_id"]
].merge(customers_df[["customer_id", "customer_unique_id"]], on="customer_id", how="left")
transacting_customers = delivered_customers["customer_unique_id"].nunique()
active_sellers = delivered_items["seller_id"].nunique()

delivery_kpi_orders = orders_df.loc[
    orders_df["order_status"].eq("delivered")
    & orders_df["delivery_days"].notna()
    & orders_df["delivery_days"].ge(0)
    & orders_df["delivery_delay_days"].notna()
]
on_time_rate_kpi = 1 - delivery_kpi_orders["is_late_delivery"].mean()
latest_reviews_kpi = (
    reviews_df.dropna(subset=["review_score"])
    .sort_values("review_answer_timestamp", na_position="first")
    .drop_duplicates(subset="order_id", keep="last")
)
high_review_share_kpi = latest_reviews_kpi["review_score"].ge(4).mean()

kpi_figure = go.Figure(
    data=[
        go.Indicator(
            mode="number",
            value=gmv,
            number={"prefix": "R$ ", "valueformat": ",.0f", "font": {"size": 36, "color": "#2563EB"}},
            title={"text": "GMV", "font": {"size": 16}},
            domain={"x": [0.0, 0.30], "y": [0.55, 1]},
        ),
        go.Indicator(
            mode="number",
            value=order_volume,
            number={"valueformat": ",.0f", "font": {"size": 36, "color": "#059669"}},
            title={"text": "Delivered Orders", "font": {"size": 16}},
            domain={"x": [0.35, 0.65], "y": [0.55, 1]},
        ),
        go.Indicator(
            mode="number",
            value=average_order_value,
            number={"prefix": "R$ ", "valueformat": ",.2f", "font": {"size": 36, "color": "#EA580C"}},
            title={"text": "Average Order Value", "font": {"size": 16}},
            domain={"x": [0.70, 1.0], "y": [0.55, 1]},
        ),
        go.Indicator(
            mode="number",
            value=transacting_customers,
            number={"valueformat": ",.0f", "font": {"size": 36, "color": "#7C3AED"}},
            title={"text": "Transacting Customers", "font": {"size": 16}},
            domain={"x": [0.00, 0.23], "y": [0, 0.35]},
        ),
        go.Indicator(
            mode="number",
            value=active_sellers,
            number={"valueformat": ",.0f", "font": {"size": 36, "color": "#DB2777"}},
            title={"text": "Active Sellers", "font": {"size": 16}},
            domain={"x": [0.25, 0.48], "y": [0, 0.35]},
        ),
        go.Indicator(
            mode="number",
            value=on_time_rate_kpi,
            number={"valueformat": ".1%", "font": {"size": 36, "color": "#0F9D78"}},
            title={"text": "On-Time Delivery", "font": {"size": 16}},
            domain={"x": [0.52, 0.75], "y": [0, 0.35]},
        ),
        go.Indicator(
            mode="number",
            value=high_review_share_kpi,
            number={"valueformat": ".1%", "font": {"size": 36, "color": "#F59E0B"}},
            title={"text": "High-Review Share", "font": {"size": 16}},
            domain={"x": [0.77, 1.00], "y": [0, 0.35]},
        ),
    ]
)

kpi_figure.update_layout(
    title="Olist Marketplace GMV and Order Scale",
    height=460,
    margin={"l": 30, "r": 30, "t": 70, "b": 30},
)
kpi_figure.show()

**Across the observed delivered orders, item-price GMV reached R$13,221,498 from September 2016 through August 2018. The dataset contains 96,478 delivered orders, 93,358 transacting customers, and 2,970 active sellers. GMV per delivered order was R$137.04; 91.9% of eligible deliveries were on time, and 77.1% of reviewed orders received four or five stars.**


### 5.2 Growth Trends and Seasonality


In [10]:
# Build monthly GMV and delivered-order time series
delivered_orders_ts = orders_df.loc[
    orders_df["order_status"].eq("delivered"),
    ["order_id", "order_purchase_timestamp"],
].copy()
delivered_orders_ts["month"] = delivered_orders_ts["order_purchase_timestamp"].dt.to_period("M").dt.to_timestamp()

monthly_orders = (
    delivered_orders_ts.groupby("month", as_index=False)
    .agg(delivered_orders=("order_id", "nunique"))
)
monthly_gmv = (
    items_df.merge(delivered_orders_ts[["order_id", "month"]], on="order_id", how="inner")
    .groupby("month", as_index=False)
    .agg(gmv=("price", "sum"))
)
monthly_market = (
    monthly_orders.merge(monthly_gmv, on="month", how="outer")
    .fillna(0)
    .sort_values("month")
)

# Calculate linear regression trend lines using actual elapsed days
elapsed_days = (monthly_market["month"] - monthly_market["month"].min()).dt.days.to_numpy()
gmv_slope, gmv_intercept = np.polyfit(elapsed_days, monthly_market["gmv"], 1)
orders_slope, orders_intercept = np.polyfit(elapsed_days, monthly_market["delivered_orders"], 1)
monthly_market["gmv_trend"] = gmv_slope * elapsed_days + gmv_intercept
monthly_market["orders_trend"] = orders_slope * elapsed_days + orders_intercept
gmv_r_squared = 1 - ((monthly_market["gmv"] - monthly_market["gmv_trend"]) ** 2).sum() / ((monthly_market["gmv"] - monthly_market["gmv"].mean()) ** 2).sum()
orders_r_squared = 1 - ((monthly_market["delivered_orders"] - monthly_market["orders_trend"]) ** 2).sum() / ((monthly_market["delivered_orders"] - monthly_market["delivered_orders"].mean()) ** 2).sum()
label_index = max(len(monthly_market) - 4, 0)
orders_axis_max = monthly_market["delivered_orders"].max() * 1.4

# Plot monthly GMV and delivered orders with dual Y-axes and regression lines
growth_figure = go.Figure()
growth_figure.add_trace(
    go.Scatter(
        x=monthly_market["month"],
        y=monthly_market["gmv"],
        mode="lines",
        name="GMV",
        line={"color": "#2563EB", "width": 3},
    )
)
growth_figure.add_trace(
    go.Scatter(
        x=monthly_market["month"],
        y=monthly_market["gmv_trend"],
        mode="lines",
        name="GMV Regression Trend",
        line={"color": "#93C5FD", "width": 2, "dash": "dash"},
    )
)
growth_figure.add_trace(
    go.Scatter(
        x=monthly_market["month"],
        y=monthly_market["delivered_orders"],
        mode="lines",
        name="Delivered Orders",
        line={"color": "#059669", "width": 3},
        yaxis="y2",
    )
)
growth_figure.add_trace(
    go.Scatter(
        x=monthly_market["month"],
        y=monthly_market["orders_trend"],
        mode="lines",
        name="Delivered Orders Regression Trend",
        line={"color": "#6EE7B7", "width": 2, "dash": "dash"},
        yaxis="y2",
    )
)
growth_figure.update_layout(
    title="Monthly GMV and Delivered Orders Growth Trend",
    xaxis={"title": "Month"},
    yaxis={"title": {"text": "GMV (R$)", "font": {"color": "#2563EB"}}, "tickfont": {"color": "#2563EB"}},
    yaxis2={"title": {"text": "Delivered Orders", "font": {"color": "#059669"}}, "tickfont": {"color": "#059669"}, "range": [0, orders_axis_max], "overlaying": "y", "side": "right"},
    template="plotly_white",
    height=550,
    legend={"orientation": "h", "y": -0.18, "x": 0.5, "xanchor": "center"},
    margin={"l": 80, "r": 85, "t": 75, "b": 110},
    annotations=[
        {"x": monthly_market["month"].iloc[label_index], "y": monthly_market["gmv_trend"].iloc[label_index], "text": f"GMV Trend<br>R² = {gmv_r_squared:.2f}", "showarrow": False, "font": {"color": "#2563EB"}, "yshift": 18},
        {"x": monthly_market["month"].iloc[label_index], "y": monthly_market["orders_trend"].iloc[label_index], "yref": "y2", "text": f"Orders Trend<br>R² = {orders_r_squared:.2f}", "showarrow": False, "font": {"color": "#059669"}, "yshift": -20},
    ],
)
growth_figure.show()

**Monthly delivered-item GMV and delivered-order counts generally rose over the observed period. August 2018 recorded approximately R$839,000 and 6,351 orders. Their fitted linear slopes were approximately R$45,700 and 334 orders per month, respectively.**


#### 5.2.1 Three-Month Multivariable Regression Forecast

This exploratory forecast uses complete months from January 2017 through August 2018. For each target, the linear model uses a year-and-month time index plus month-of-year seasonality:

$$
y_m=\beta_0+\beta_1[12(\mathrm{Year}_m-2017)+(\mathrm{Month}_m-1)]
+\beta_2\sin\!\left(\frac{2\pi(\mathrm{Month}_m-1)}{12}\right)
+\beta_3\cos\!\left(\frac{2\pi(\mathrm{Month}_m-1)}{12}\right)+\varepsilon_m.
$$

The shaded 95% confidence interval describes uncertainty in the **expected monthly mean**, not a range guaranteed to contain an individual future month. With only 20 historical months, trend changes, autocorrelation, and external shocks are not captured.

In [11]:
# Forecast three months from the last complete observed month.
from plotly.subplots import make_subplots
from scipy.stats import t as student_t

forecast_history = monthly_market.loc[
    monthly_market["month"].between(pd.Timestamp("2017-01-01"), pd.Timestamp("2018-08-01"))
].copy()
expected_months = pd.date_range("2017-01-01", "2018-08-01", freq="MS")
assert forecast_history["month"].tolist() == expected_months.tolist(), "Training months must be complete and consecutive."
forecast_months = pd.date_range(
    forecast_history["month"].max() + pd.offsets.MonthBegin(1), periods=3, freq="MS"
)

def monthly_design(months):
    year = months.dt.year.to_numpy()
    month = months.dt.month.to_numpy()
    elapsed_months = (year - 2017) * 12 + (month - 1)
    return np.column_stack([
        np.ones(len(months)), elapsed_months,
        np.sin(2 * np.pi * (month - 1) / 12),
        np.cos(2 * np.pi * (month - 1) / 12),
    ])

history_design = monthly_design(forecast_history["month"])
future_design = monthly_design(pd.Series(forecast_months))
forecast_table = pd.DataFrame({"month": forecast_months})

for target in ["gmv", "delivered_orders"]:
    observed = forecast_history[target].to_numpy(dtype=float)
    coefficients = np.linalg.lstsq(history_design, observed, rcond=None)[0]
    fitted = history_design @ coefficients
    residual_degrees = len(observed) - history_design.shape[1]
    residual_variance = np.sum((observed - fitted) ** 2) / residual_degrees
    inverse_information = np.linalg.pinv(history_design.T @ history_design)
    expected_values = future_design @ coefficients
    mean_standard_error = np.sqrt(
        residual_variance * np.einsum("ij,jk,ik->i", future_design, inverse_information, future_design)
    )
    confidence_margin = student_t.ppf(0.975, residual_degrees) * mean_standard_error
    forecast_table[target] = expected_values
    forecast_table[f"{target}_ci_low"] = expected_values - confidence_margin
    forecast_table[f"{target}_ci_high"] = expected_values + confidence_margin

forecast_figure = make_subplots(
    rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.12,
    subplot_titles=("GMV", "Delivered Orders"),
)
for row, target, color, fill_color in [
    (1, "gmv", "#2563EB", "rgba(37,99,235,0.17)"),
    (2, "delivered_orders", "#059669", "rgba(5,150,105,0.17)"),
]:
    forecast_figure.add_trace(go.Scatter(
        x=forecast_history["month"], y=forecast_history[target],
        mode="lines+markers", name=f"Observed {target.replace('_', ' ').title()}",
        line={"color": color, "width": 2.5},
    ), row=row, col=1)
    forecast_figure.add_trace(go.Scatter(
        x=forecast_months, y=forecast_table[f"{target}_ci_low"],
        mode="lines", line={"color": color, "width": 0}, showlegend=False, hoverinfo="skip",
    ), row=row, col=1)
    forecast_figure.add_trace(go.Scatter(
        x=forecast_months, y=forecast_table[f"{target}_ci_high"],
        mode="lines", line={"color": color, "width": 0}, fill="tonexty",
        fillcolor=fill_color, name="95% Mean CI" if row == 1 else None,
        showlegend=row == 1, hoverinfo="skip",
    ), row=row, col=1)
    forecast_figure.add_trace(go.Scatter(
        x=[forecast_history["month"].iloc[-1], *forecast_months],
        y=[forecast_history[target].iloc[-1], *forecast_table[target].tolist()],
        mode="lines+markers", name=f"Forecast {target.replace('_', ' ').title()}",
        line={"color": color, "width": 3, "dash": "dash"},
        marker={"symbol": "diamond", "size": 8},
    ), row=row, col=1)
forecast_figure.update_yaxes(title_text="GMV (R$)", row=1, col=1)
forecast_figure.update_yaxes(title_text="Orders", row=2, col=1)
forecast_figure.update_xaxes(title_text="Month", row=2, col=1)
forecast_figure.update_layout(
    title="Three-Month Historical Forecast from August 2018",
    template="plotly_white", height=760,
    legend={"orientation": "h", "y": 1.08}, margin={"l": 75, "r": 35, "t": 110, "b": 65},
)
forecast_figure.show()
display(forecast_table.style.format({
    "gmv": "R$ {:,.0f}", "gmv_ci_low": "R$ {:,.0f}", "gmv_ci_high": "R$ {:,.0f}",
    "delivered_orders": "{:,.0f}", "delivered_orders_ci_low": "{:,.0f}",
    "delivered_orders_ci_high": "{:,.0f}",
}))

,month,gmv,gmv_ci_low,gmv_ci_high,delivered_orders,delivered_orders_ci_low,delivered_orders_ci_high
0,2018-09-01 00:00:00,"R$ 1,099,896","R$ 960,680","R$ 1,239,113","8,088","7,038","9,139"
1,2018-10-01 00:00:00,"R$ 1,177,456","R$ 1,019,997","R$ 1,334,916","8,708","7,520","9,896"
2,2018-11-01 00:00:00,"R$ 1,252,709","R$ 1,078,870","R$ 1,426,548","9,293","7,982","10,605"


**The exploratory model projects September–November 2018 GMV of approximately R$1.10 million, R$1.18 million, and R$1.25 million, alongside 8,088, 8,708, and 9,293 delivered orders. The plotted 95% bands are intervals for the model's expected monthly means.**


### 5.3 Market Structure and Seller Competition


In [12]:
# Prepare delivered-order item data for GMV market-share analysis
delivered_order_ids = orders_df.loc[orders_df["order_status"].eq("delivered"), "order_id"]
market_items = items_df[items_df["order_id"].isin(delivered_order_ids)].copy()

# Plot product-category market share by GMV
category_share = (
    market_items.merge(products_df[["product_id", "product_category_name"]], on="product_id", how="left")
    .merge(translation_df, on="product_category_name", how="left")
    .assign(category_name=lambda dataframe: dataframe["product_category_name_english"].fillna(dataframe["product_category_name"]).fillna("Unknown"))
    .groupby("category_name", as_index=False)["price"]
    .sum()
    .rename(columns={"price": "gmv"})
    .sort_values("gmv", ascending=False)
)
category_share["market_share"] = category_share["gmv"] / category_share["gmv"].sum()

# Show every category group, including Unknown, without merging the long tail.
treemap_view = category_share.copy()
treemap_view["plot_label"] = treemap_view["category_name"].str.replace("_", " ").str.title()
category_treemap = px.treemap(
    treemap_view, path=["plot_label"], values="gmv", color="gmv",
    color_continuous_scale="Blues", custom_data=["market_share"],
    title="Olist Delivered-Item GMV Share by Category",
)
category_treemap.update_traces(marker_line_width=0, marker_line_color="#F8FAFC", tiling_pad=3, root_color="#F8FAFC", pathbar_visible=False)
category_treemap.update_traces(
    texttemplate="%{label}<br>%{customdata[0]:.1%}",
    hovertemplate="<b>%{label}</b><br>GMV: R$ %{value:,.0f}<br>Olist share: %{customdata[0]:.2%}<extra></extra>",
)
category_treemap.update_layout(height=750, uniformtext={"minsize": 10, "mode": "hide"}, margin={"t": 70, "l": 20, "r": 20, "b": 20})
category_treemap.show()

**Health and Beauty, Watches and Gifts, Bed Bath Table, Sports Leisure, Computers Accessories, and Furniture Decor are the six largest named categories by delivered-item GMV. Together, they account for 45.2% of total delivered-item GMV, including the Unknown category in the denominator. The treemap also shows a long tail of smaller categories; the separate Pareto chart quantifies this concentration among named categories.**


In [13]:
# Build a product-category Pareto table from delivered-order GMV
category_pareto = category_share.loc[category_share["category_name"].ne("Unknown")].sort_values("gmv", ascending=False).reset_index(drop=True).copy()
category_pareto["cumulative_share"] = category_pareto["gmv"].cumsum() / category_pareto["gmv"].sum()
pareto_80_category_count = int(category_pareto["cumulative_share"].ge(0.80).idxmax() + 1)
pareto_80_category_percentage = pareto_80_category_count / len(category_pareto)

# Show the ranked categories through the 80% crossover; keep the full table above.
pareto_view = category_pareto.head(pareto_80_category_count).copy()
pareto_view["plot_label"] = pareto_view["category_name"].str.replace("_", " ").str.title()
pareto_view["cumulative_share"] = pareto_view["gmv"].cumsum() / category_pareto["gmv"].sum()
pareto_view = pareto_view.iloc[::-1]
category_pareto_figure = go.Figure()
category_pareto_figure.add_trace(go.Bar(
    x=pareto_view["gmv"], y=pareto_view["plot_label"], orientation="h",
    name="Category GMV", marker_color="#2563EB",
    hovertemplate="<b>%{y}</b><br>GMV: R$ %{x:,.0f}<extra></extra>",
))
category_pareto_figure.add_trace(go.Scatter(
    x=pareto_view["cumulative_share"], y=pareto_view["plot_label"],
    xaxis="x2", mode="lines+markers", name="Cumulative Classified GMV Share",
    line={"color": "#EA580C", "width": 2.5}, marker={"size": 6},
    hovertemplate="<b>%{y}</b><br>Cumulative share: %{x:.1%}<extra></extra>",
))
category_pareto_figure.update_layout(
    title=f"Olist GMV Pareto: {pareto_80_category_percentage:.1%} of Named Categories Reach 80% of Classified GMV",
    xaxis={"title": "Delivered-Item GMV (R$)", "rangemode": "tozero"},
    xaxis2={"title": "Cumulative Classified GMV Share", "overlaying": "x", "side": "top",
            "range": [0, 1.05], "tickformat": ".0%",
            "tickvals": [0, 0.2, 0.4, 0.6, 0.8, 1.0]},
    yaxis={"title": "Product Category", "automargin": True},
    template="plotly_white", height=850,
    legend={"orientation": "h", "y": -0.10, "x": 0.5, "xanchor": "center"},
    shapes=[{"type": "line", "xref": "x2", "x0": 0.8, "x1": 0.8,
             "yref": "paper", "y0": 0, "y1": 1,
             "line": {"color": "#D97706", "width": 2, "dash": "dash"}}],
    margin={"t": 110, "l": 200, "r": 55, "b": 85},
)
category_pareto_figure.show()

**The top 17 of 73 named product categories—23.3% of classified categories—account for approximately 80% of delivered-item GMV among products with a known category. This is a roughly Pareto-like distribution: GMV is concentrated in a relatively small group of categories, followed by a long tail. The Unknown group is excluded from the Pareto calculation but remains visible in the treemap.**


### 5.4 Payment Behavior and Consumer Spending


In [14]:
from plotly.subplots import make_subplots

# Exclude undefined payment records from the payment analysis
valid_payments = payments_df.loc[
    payments_df["payment_type"].fillna("").str.lower().ne("not_defined")
].copy()

# Summarize total payment value across all defined payment methods
payment_behavior = (
    valid_payments.groupby("payment_type", as_index=False)["payment_value"]
    .sum()
    .sort_values("payment_value", ascending=False)
)
payment_behavior["payment_method"] = (
    payment_behavior["payment_type"].str.replace("_", " ", regex=False).str.title()
)

# Calculate period-normalized monthly and annual spend per transacting consumer
consumer_payments = (
    valid_payments[["order_id", "payment_value"]]
    .merge(
        orders_df[["order_id", "customer_id", "order_purchase_timestamp"]],
        on="order_id",
        how="inner",
    )
    .merge(
        customers_df[["customer_id", "customer_unique_id"]],
        on="customer_id",
        how="inner",
    )
    .dropna(subset=["customer_unique_id", "order_purchase_timestamp", "payment_value"])
)
consumer_payments["purchase_month"] = consumer_payments["order_purchase_timestamp"].dt.to_period("M")
consumer_payments["purchase_year"] = consumer_payments["order_purchase_timestamp"].dt.year

# Use 2017 because it is the only complete calendar year in the dataset
consumer_payments_2017 = consumer_payments.loc[consumer_payments["purchase_year"].eq(2017)].copy()
monthly_consumer_spend = (
    consumer_payments_2017.groupby(["customer_unique_id", "purchase_month"])["payment_value"]
    .sum()
)
annual_consumer_spend = (
    consumer_payments_2017.groupby("customer_unique_id")["payment_value"]
    .sum()
)
average_monthly_consumer_spend = monthly_consumer_spend.mean()
average_annual_consumer_spend = annual_consumer_spend.mean()
consumer_order_counts = consumer_payments_2017.groupby("customer_unique_id")["order_id"].nunique()
repeat_purchase_rate = consumer_order_counts.gt(1).mean()

# Combine the payment-method chart with three vertically stacked consumer indicators
payment_figure = make_subplots(
    rows=3,
    cols=2,
    specs=[
        [{"type": "xy", "rowspan": 3}, {"type": "domain"}],
        [None, {"type": "domain"}],
        [None, {"type": "domain"}],
    ],
    column_widths=[0.68, 0.32],
    row_heights=[1 / 3, 1 / 3, 1 / 3],
    horizontal_spacing=0.12,
    vertical_spacing=0.08,
)

payment_colors = ["#2563EB", "#0F766E", "#F59E0B", "#7C3AED", "#64748B"]
payment_figure.add_trace(
    go.Bar(
        x=payment_behavior["payment_method"],
        y=payment_behavior["payment_value"],
        marker={"color": payment_colors[:len(payment_behavior)]},
        text=payment_behavior["payment_value"],
        texttemplate="R$ %{text:,.0f}",
        textposition="outside",
        cliponaxis=False,
        hovertemplate="<b>%{x}</b><br>Total Payment Value: R$ %{y:,.2f}<extra></extra>",
        showlegend=False,
    ),
    row=1,
    col=1,
)

payment_figure.add_trace(
    go.Indicator(
        mode="number",
        value=average_monthly_consumer_spend,
        number={"prefix": "R$ ", "valueformat": ",.2f", "font": {"size": 38, "color": "#2563EB"}},
        title={"text": "2017 Average Monthly Spend<br>per Active Consumer", "font": {"size": 16, "color": "#334155"}},
    ),
    row=1,
    col=2,
)
payment_figure.add_trace(
    go.Indicator(
        mode="number",
        value=average_annual_consumer_spend,
        number={"prefix": "R$ ", "valueformat": ",.2f", "font": {"size": 38, "color": "#EA580C"}},
        title={"text": "2017 Average Annual Spend<br>per Active Consumer", "font": {"size": 16, "color": "#334155"}},
    ),
    row=2,
    col=2,
)
payment_figure.add_trace(
    go.Indicator(
        mode="number",
        value=repeat_purchase_rate,
        number={"valueformat": ".1%", "font": {"size": 38, "color": "#7C3AED"}},
        title={"text": "2017 Repeat Purchase Rate<br>among Active Consumers", "font": {"size": 16, "color": "#334155"}},
    ),
    row=3,
    col=2,
)

payment_figure.update_layout(
    title="Payment Behavior and Consumer Spending",
    template="plotly_white",
    height=650,
    bargap=0.35,
    margin={"l": 70, "r": 35, "t": 85, "b": 70},
    font={"family": "Arial", "color": "#334155"},
    shapes=[
        {"type": "rect", "xref": "paper", "yref": "paper", "x0": 0.73, "x1": 1.00, "y0": 0.70, "y1": 1.00, "fillcolor": "#EFF6FF", "line": {"width": 0}, "layer": "below"},
        {"type": "rect", "xref": "paper", "yref": "paper", "x0": 0.73, "x1": 1.00, "y0": 0.35, "y1": 0.65, "fillcolor": "#FFF7ED", "line": {"width": 0}, "layer": "below"},
        {"type": "rect", "xref": "paper", "yref": "paper", "x0": 0.73, "x1": 1.00, "y0": 0.00, "y1": 0.30, "fillcolor": "#F5F3FF", "line": {"width": 0}, "layer": "below"},
    ],
)
payment_figure.update_xaxes(title_text="Payment Method", row=1, col=1)
payment_figure.update_yaxes(title_text="Total Payment Value (R$)", tickprefix="R$ ", tickformat=",.0f", rangemode="tozero", row=1, col=1)
payment_figure.show()


**Credit cards account for the largest observed payment value, followed by boleto, vouchers, and debit cards. In 2017, average spending was R$163.51 per active consumer-month and R$165.85 per consumer who purchased at least once during the year. These figures use different denominators: the monthly measure averages months in which a consumer was active, while the annual measure averages each active consumer's full-year spending. The observed repeat-purchase rate for 2017 was 2.87%.**


## 6. Product Analysis

Category comparisons combine observed prices, buyer-charged freight, product breadth, seller concentration, and comparable-month GMV.


### 6.1 Category Prices and Buyer-Charged Freight

In [15]:
# Prepare category-level delivered-order data for product analyses
delivered_order_details = orders_df.loc[
    orders_df["order_status"].eq("delivered"),
    ["order_id", "customer_id", "order_purchase_timestamp", "delivery_days", "delivery_delay_days", "is_late_delivery"],
].copy()
product_sales = (
    items_df[items_df["order_id"].isin(delivered_order_details["order_id"])]
    .merge(products_df[["product_id", "product_category_name"]], on="product_id", how="left")
    .merge(translation_df, on="product_category_name", how="left")
    .merge(delivered_order_details[["order_id", "order_purchase_timestamp"]], on="order_id", how="left")
)
product_sales["category_name"] = (
    product_sales["product_category_name_english"]
    .fillna(product_sales["product_category_name"])
    .fillna("Unknown")
)
category_performance = (
    product_sales.groupby("category_name", as_index=False)
    .agg(
        gmv=("price", "sum"),
        orders=("order_id", "nunique"),
        units=("order_item_id", "count"),
        avg_item_price=("price", "mean"),
    )
)
category_performance["gmv_share"] = category_performance["gmv"] / category_performance["gmv"].sum()



In [16]:
# Compare category pricing with freight burden
category_economics = (
    product_sales.loc[product_sales["price"].gt(0)]
    .groupby("category_name", as_index=False)
    .agg(
        items=("order_item_id", "count"),
        median_price=("price", "median"),
        average_price=("price", "mean"),
        average_freight=("freight_value", "mean"),
        total_price=("price", "sum"),
        total_freight=("freight_value", "sum"),
    )
)
category_economics["freight_to_price_ratio"] = category_economics["total_freight"] / category_economics["total_price"]
category_economics_view = category_economics.loc[category_economics["items"].ge(100)].copy()
pricing_figure = px.scatter(
    category_economics_view,
    x="median_price",
    y="freight_to_price_ratio",
    size="total_price",
    color="average_freight",
    hover_name="category_name",
    color_continuous_scale="Blues",
    title="Category Pricing and Freight Burden",
    labels={"median_price": "Median Item Price (R$)", "freight_to_price_ratio": "Freight-to-Price Ratio", "average_freight": "Average Freight (R$)", "total_price": "Category GMV (R$)"},
)

pricing_figure.update_yaxes(tickformat=".0%", rangemode="tozero")
pricing_figure.update_xaxes(rangemode="tozero")
pricing_figure.update_layout(template="plotly_white", height=560)
pricing_figure.show()


**Among categories with at least 100 observed items, median item prices range from approximately R$22 to R$1,100, while buyer-charged freight equals approximately 4.4%–36.5% of item-price GMV, depending on category. Computers has the highest displayed median price and one of the lowest freight-to-price ratios. These measures describe prices and freight charged to buyers, not seller costs or profit margins.**


### 6.2 Product Portfolio and Seller Competitiveness


In [17]:
# Measure product breadth, seller coverage, and seller concentration by category
seller_category_gmv = (
    product_sales.groupby(["category_name", "seller_id"], as_index=False)["price"]
    .sum()
    .rename(columns={"price": "seller_gmv"})
)
seller_category_gmv["seller_share"] = seller_category_gmv["seller_gmv"] / seller_category_gmv.groupby("category_name")["seller_gmv"].transform("sum")
seller_concentration = (
    seller_category_gmv.assign(squared_share=lambda dataframe: dataframe["seller_share"] ** 2)
    .groupby("category_name", as_index=False)["squared_share"]
    .sum()
    .rename(columns={"squared_share": "seller_hhi"})
)
category_portfolio = (
    product_sales.groupby("category_name", as_index=False)
    .agg(gmv=("price", "sum"), unique_products=("product_id", "nunique"), unique_sellers=("seller_id", "nunique"), orders=("order_id", "nunique"))
    .merge(seller_concentration, on="category_name", how="left")
)
portfolio_figure = px.scatter(
    category_portfolio.loc[category_portfolio["orders"].ge(50)],
    x="unique_sellers",
    y="unique_products",
    size="gmv",
    color="seller_hhi",
    hover_name="category_name",
    color_continuous_scale="Oranges",
    log_x=True,
    title="Product Portfolio Breadth and Seller Competition",
    labels={"unique_sellers": "Unique Sellers (log scale)", "unique_products": "Unique Products", "gmv": "Category GMV (R$)", "seller_hhi": "Seller HHI"},
)

portfolio_figure.update_layout(template="plotly_white", height=580)
portfolio_figure.show()


$$
\mathrm{HHI}_c = \sum_{s \in c} \left(\frac{\mathrm{GMV}_{s,c}}{\mathrm{GMV}_c}\right)^2
$$

Here, \(s\) is a seller in category \(c\). A value near 1 indicates that one seller dominates delivered-item GMV; a lower value indicates a more distributed seller mix within Olist.

**The chart compares distinct products, active sellers, and seller-GMV concentration by category. Computers records an HHI of about 0.58, indicating a more concentrated seller mix than many of the displayed categories.**


### 6.3 Monthly Category Demand

The five categories are selected by delivered-item GMV across the observed period and then held fixed for the monthly comparison. The chart reports Olist item-price GMV from January 2017 to August 2018. The table compares January–August 2018 with the same eight months in 2017.

In [18]:
# Compare a fixed set of leading categories across months and equal-length year windows.
category_monthly = product_sales.loc[
    product_sales["order_purchase_timestamp"].between(
        pd.Timestamp("2017-01-01"), pd.Timestamp("2018-08-31 23:59:59")
    )
].copy()
category_monthly["month"] = category_monthly["order_purchase_timestamp"].dt.to_period("M").dt.to_timestamp()
leading_categories = category_performance.nlargest(5, "gmv")["category_name"].tolist()
category_monthly = (
    category_monthly.loc[category_monthly["category_name"].isin(leading_categories)]
    .groupby(["month", "category_name"], as_index=False)
    .agg(gmv=("price", "sum"), delivered_orders=("order_id", "nunique"))
)
category_monthly["category_label"] = category_monthly["category_name"].str.replace("_", " ").str.title()
category_growth_figure = px.line(
    category_monthly, x="month", y="gmv", color="category_label",
    markers=True, hover_data={"delivered_orders": ":,", "gmv": ":,.0f"},
    title="Monthly Olist Delivered-Item GMV: Five Leading Categories",
    labels={"month": "Purchase Month", "gmv": "Item-Price GMV (R$)",
            "category_label": "Category"},
    color_discrete_sequence=["#2563EB", "#0F766E", "#EA580C", "#7C3AED", "#D97706"],
)
category_growth_figure.update_layout(
    template="plotly_white", height=580,
    legend={"orientation": "h", "y": -0.22, "x": 0.5, "xanchor": "center"},
    margin={"l": 80, "r": 40, "t": 80, "b": 140},
)
category_growth_figure.show()

equal_window_sales = product_sales.loc[
    product_sales["order_purchase_timestamp"].dt.year.isin([2017, 2018])
    & product_sales["order_purchase_timestamp"].dt.month.le(8)
    & product_sales["category_name"].isin(leading_categories)
].copy()
equal_window_sales["year"] = equal_window_sales["order_purchase_timestamp"].dt.year
category_growth_comparison = (
    equal_window_sales.groupby(["category_name", "year"])["price"]
    .sum().unstack("year", fill_value=0)
    .rename(columns={2017: "GMV Jan-Aug 2017", 2018: "GMV Jan-Aug 2018"})
)
category_growth_comparison["Change"] = (
    category_growth_comparison["GMV Jan-Aug 2018"]
    / category_growth_comparison["GMV Jan-Aug 2017"].replace(0, np.nan) - 1
)
display(category_growth_comparison.sort_values("GMV Jan-Aug 2018", ascending=False).round(3))

year,GMV Jan-Aug 2017,GMV Jan-Aug 2018,Change
category_name,,,
health_beauty,243521.70,755724.50,2.103
watches_gifts,201137.86,687855.20,2.420
bed_bath_table,254444.82,532358.85,1.092
sports_leisure,216724.33,517166.26,1.386
computers_accessories,214631.27,496269.30,1.312


**The five displayed categories each generated more delivered-item GMV in January–August 2018 than in the same months of 2017. The equal-window increases range from 109.2% for Bed Bath Table to 242.0% for Watches Gifts.**


## 7. Customer Analysis

Customer-level measures connect transaction frequency and spending with RFM groups, value tiers, and equal-follow-up repeat-purchase cohorts.


### 7.1 Customer Base and Purchase Behavior


In [19]:
# Build a delivered-order customer table at one row per order
order_payments = payments_df.groupby("order_id", as_index=False)["payment_value"].sum()
delivered_customer_orders = (
    delivered_order_details.merge(customers_df[["customer_id", "customer_unique_id"]], on="customer_id", how="left")
    .merge(order_payments, on="order_id", how="left")
    .dropna(subset=["customer_unique_id"])
)
customer_activity = (
    delivered_customer_orders.groupby("customer_unique_id", as_index=False)
    .agg(order_count=("order_id", "nunique"), total_spend=("payment_value", "sum"), first_purchase=("order_purchase_timestamp", "min"), last_purchase=("order_purchase_timestamp", "max"))
)
customer_activity["repeat_customer"] = customer_activity["order_count"].gt(1)
frequency_thresholds = [1, 2, 3, 4, 5]
frequency_reach = pd.DataFrame({
    "stage": [f"At least {threshold} delivered order{'s' if threshold > 1 else ''}" for threshold in frequency_thresholds],
    "consumers": [(customer_activity["order_count"] >= threshold).sum() for threshold in frequency_thresholds],
})
frequency_reach["share_of_buyers"] = frequency_reach["consumers"] / frequency_reach["consumers"].iloc[0]
frequency_reach["log_consumers"] = np.log10(frequency_reach["consumers"] + 1)
frequency_figure = go.Figure(go.Funnel(
    y=frequency_reach["stage"], x=frequency_reach["log_consumers"],
    customdata=frequency_reach[["consumers", "share_of_buyers"]].to_numpy(),
    texttemplate="%{customdata[0]:,.0f} consumers · %{customdata[1]:.1%} of buyers",
    textposition="inside", marker={"color": ["#1D4ED8", "#2563EB", "#3B82F6", "#60A5FA", "#93C5FD"]},
    hovertemplate="%{y}<br>%{customdata[0]:,.0f} consumers<br>%{customdata[1]:.2%} of buyers<extra></extra>",
))
frequency_figure.update_layout(title="Delivered-Order Reach by Purchase Threshold", template="plotly_white", height=520, margin={"l": 210, "r": 40, "t": 80, "b": 70})
frequency_figure.update_xaxes(title="log10(Consumers + 1)")
frequency_figure.show()


**Among 93,358 customers with a delivered order, 2,801 (3.00%) placed at least two delivered orders in the available records, and 47 (0.05%) placed four or more. Section 7.4 uses equal follow-up windows to compare new-customer cohorts.**


### 7.2 RFM Segmentation and Repeat Purchasing


In [20]:
# Create RFM scores from delivered-order customer activity
rfm_reference_date = delivered_customer_orders["order_purchase_timestamp"].max().normalize() + pd.Timedelta(days=1)
rfm = customer_activity[["customer_unique_id", "order_count", "total_spend", "last_purchase"]].copy()
rfm["recency_days"] = (rfm_reference_date - rfm["last_purchase"]).dt.days
rfm["R"] = pd.qcut(rfm["recency_days"].rank(method="first"), 4, labels=[4, 3, 2, 1]).astype(int)
rfm["F"] = pd.cut(rfm["order_count"], bins=[0, 1, 2, 3, np.inf], labels=[1, 2, 3, 4]).astype(int)
rfm["M"] = pd.qcut(rfm["total_spend"].rank(method="first"), 4, labels=[1, 2, 3, 4]).astype(int)
rfm["segment"] = np.select(
    [
        rfm["R"].ge(3) & rfm["F"].ge(3) & rfm["M"].ge(3),
        rfm["R"].ge(2) & rfm["F"].ge(2),
        rfm["R"].ge(3) & rfm["F"].eq(1),
        rfm["R"].le(2) & rfm["F"].ge(2),
    ],
    ["Champions", "Loyal Customers", "Promising One-Time Buyers", "At-Risk Repeat Buyers"],
    default="Other One-Time Buyers",
)


In [21]:
rfm_quadrant = (
    rfm.groupby("segment", as_index=False)
    .agg(customers=("customer_unique_id", "nunique"), average_recency_days=("recency_days", "mean"), average_orders=("order_count", "mean"), average_spend=("total_spend", "mean"))
)
rfm_quadrant["customer_share"] = rfm_quadrant["customers"] / rfm_quadrant["customers"].sum()
rfm_short_labels = {"Champions": "Champions", "Loyal Customers": "Loyal", "Promising One-Time Buyers": "Promising One-Time", "At-Risk Repeat Buyers": "At-Risk Repeat", "Other One-Time Buyers": "Other One-Time"}
rfm_quadrant["bubble_label"] = [f"{rfm_short_labels[segment]}<br>{customers:,} ({share:.1%})" for segment, customers, share in zip(rfm_quadrant["segment"], rfm_quadrant["customers"], rfm_quadrant["customer_share"])]
rfm_recency_cutoff = rfm["recency_days"].median()
rfm_repeat_cutoff = 1.5
rfm_quadrant_figure = px.scatter(
    rfm_quadrant, x="average_recency_days", y="average_orders",
    size="customers", color="segment", text="bubble_label", size_max=58,
    color_discrete_map={"Champions": "#0F766E", "Loyal Customers": "#2563EB", "Promising One-Time Buyers": "#F59E0B", "At-Risk Repeat Buyers": "#EA580C", "Other One-Time Buyers": "#64748B"},
    hover_data={"customers": ":,", "customer_share": ":.1%", "average_spend": ":.2f", "average_recency_days": ":.1f", "average_orders": ":.2f", "bubble_label": False},
    title="RFM Segment Quadrant: Recency vs Purchase Frequency",
    labels={"average_recency_days": "Average Days Since Last Purchase (lower is more recent)", "average_orders": "Average Delivered Orders per Consumer"},
)
rfm_quadrant_figure.add_vline(x=rfm_recency_cutoff, line_dash="dash", line_color="#94A3B8")
rfm_quadrant_figure.add_hline(y=rfm_repeat_cutoff, line_dash="dash", line_color="#94A3B8")
rfm_quadrant_figure.update_traces(textposition="top center", textfont_size=12, marker_line_width=1, marker_line_color="white")
rfm_quadrant_figure.update_layout(template="plotly_white", height=650, showlegend=False, margin={"l": 80, "r": 95, "t": 90, "b": 80})
rfm_quadrant_figure.show()

**The stated RFM rules divide observed buyers into five segments. Promising One-Time Buyers represent 48.35% of buyers, while Loyal Customers represent 2.23% and Champions 0.15%. The large promising one-time segment provides a clearly defined population for testing first-to-second-purchase retention efforts; the chart alone does not establish which intervention would work.**


### 7.3 Customer Value and Consumption Level


In [22]:
# Segment consumers by total delivered-order spend percentiles
customer_value = customer_activity.copy()
customer_value["value_tier"] = pd.qcut(
    customer_value["total_spend"].rank(method="first"),
    q=[0, 0.50, 0.80, 0.95, 1.00],
    labels=["Low Value", "Mid Value", "High Value", "Premium"],
)
value_tier_summary = (
    customer_value.groupby("value_tier", observed=False, as_index=False)
    .agg(customers=("customer_unique_id", "nunique"), spend=("total_spend", "sum"), average_orders=("order_count", "mean"))
)
value_tier_summary["customer_share"] = value_tier_summary["customers"] / value_tier_summary["customers"].sum()
value_tier_summary["spend_share"] = value_tier_summary["spend"] / value_tier_summary["spend"].sum()
value_tier_summary["spend_to_customer_ratio"] = value_tier_summary["spend_share"] / value_tier_summary["customer_share"]
value_figure = make_subplots(specs=[[{"secondary_y": True}]])
value_figure.add_trace(go.Bar(x=value_tier_summary["value_tier"], y=value_tier_summary["customer_share"], name="Customer Share", marker_color="#93C5FD", text=value_tier_summary["customer_share"], texttemplate="%{text:.1%}", textposition="outside"), secondary_y=False)
value_figure.add_trace(go.Bar(x=value_tier_summary["value_tier"], y=value_tier_summary["spend_share"], name="Spend Share", marker_color="#2563EB", text=value_tier_summary["spend_share"], texttemplate="%{text:.1%}", textposition="outside"), secondary_y=False)
value_figure.add_trace(go.Scatter(x=value_tier_summary["value_tier"], y=value_tier_summary["spend_to_customer_ratio"], name="Spend Share / Customer Share", mode="lines+markers+text", line={"color": "#EA580C", "width": 3}, marker={"size": 9}, text=[f"{ratio:.2f}×" for ratio in value_tier_summary["spend_to_customer_ratio"]], textposition="top center"), secondary_y=True)
value_figure.update_yaxes(title_text="Share of Total", tickformat=".0%", range=[0, value_tier_summary[["customer_share", "spend_share"]].max().max() * 1.25], secondary_y=False)
value_figure.update_yaxes(title_text="Spend Share / Customer Share (×)", rangemode="tozero", secondary_y=True)
value_figure.update_layout(title="Customer Value Tier Composition and Share Ratio", template="plotly_white", height=550, barmode="group", xaxis_title="Customer Value Tier", legend={"orientation": "h", "y": 1.12, "x": 0.5, "xanchor": "center"}, margin={"l": 70, "r": 85, "t": 120, "b": 60})
value_figure.show()


$$Q_t = \frac{S_t}{C_t}$$

Here, $C_t$ is the share of customers in value tier $t$, and $S_t$ is that tier's share of total observed spending. A ratio above 1 indicates a larger spending share than customer share.

**Low Value customers account for 50% of buyers and 19.2% of spending; Mid Value customers account for 30% and 27.2%; High Value customers account for 15% and 26.7%; and Premium customers account for 5% and 26.8%. The top two tiers comprise 20% of customers and contribute 53.5% of observed spending. Average spending per customer is approximately 2.36 times higher in Mid Value than Low Value, 1.96 times higher in High Value than Mid Value, and 3.01 times higher in Premium than High Value.**


In [23]:
value_tier_repeat = (
    customer_value.groupby("value_tier", observed=False, as_index=False)
    .agg(repeat_rate=("repeat_customer", "mean"))
)
value_quadrant = value_tier_summary.merge(value_tier_repeat, on="value_tier")
value_quadrant["average_spend"] = value_quadrant["spend"] / value_quadrant["customers"]
value_quadrant["customer_share"] = value_quadrant["customers"] / value_quadrant["customers"].sum()
value_quadrant["bubble_label"] = [f"{tier}<br>{customers:,} ({share:.1%})" for tier, customers, share in zip(value_quadrant["value_tier"], value_quadrant["customers"], value_quadrant["customer_share"])]
value_spend_cutoff = customer_value["total_spend"].mean()
value_repeat_cutoff = customer_value["repeat_customer"].mean()
value_quadrant_figure = px.scatter(
    value_quadrant, x="average_spend", y="repeat_rate",
    size="customers", color="value_tier", text="bubble_label", size_max=58,
    color_discrete_map={"Low Value": "#93C5FD", "Mid Value": "#2563EB", "High Value": "#7C3AED", "Premium": "#EA580C"},
    hover_data={"customers": ":,", "customer_share": ":.1%", "average_orders": ":.2f", "average_spend": ":.2f", "repeat_rate": ":.2%", "bubble_label": False},
    title="Customer Value Quadrant: Spending vs Repeat Purchase",
    labels={"average_spend": "Average Delivered-Order Spend per Consumer (R$)", "repeat_rate": "Repeat Purchase Rate", "value_tier": "Customer Value Tier"},
)
value_quadrant_figure.add_vline(x=value_spend_cutoff, line_dash="dash", line_color="#94A3B8")
value_quadrant_figure.add_hline(y=value_repeat_cutoff, line_dash="dash", line_color="#94A3B8")
value_quadrant_figure.update_traces(textposition="top center", textfont_size=12, marker_line_width=1, marker_line_color="white")
value_quadrant_figure.update_yaxes(tickformat=".0%")
value_quadrant_figure.update_layout(template="plotly_white", height=630, showlegend=False, margin={"l": 75, "r": 95, "t": 90, "b": 80})
value_quadrant_figure.show()

**The four spend-ranked groups show rising observed repeat-purchase rates: 0.7% for Low Value, 3.3% for Mid Value, 7.9% for High Value, and 9.2% for Premium customers. Mid Value customers average R$149.97 in delivered-order spend and sit near the overall spending and repeat-rate reference lines. Bubble distance describes these group averages; it does not measure how easily customers can move between groups.**


### 7.4 Six-Month Repeat-Purchase Cohorts

The cohort month is each consumer's first observed delivered-order purchase month. Only January–December 2017 first-purchase cohorts are shown, and every cohort has at least six subsequent calendar months in the dataset. A cell reports the share placing a delivered order in that exact month after the first purchase; it is not a cumulative retention rate. Additional orders in the same calendar month as the first purchase are excluded from the subsequent-month rate. Purchases outside Olist and repeat purchases after the six-month window are unobserved.

In [24]:
# Keep cohort follow-up comparable at six calendar months for 2017 first-time buyers.
cohort_orders = delivered_customer_orders[[
    "customer_unique_id", "order_id", "order_purchase_timestamp"
]].dropna().drop_duplicates(subset="order_id").copy()
cohort_orders["purchase_month"] = cohort_orders["order_purchase_timestamp"].dt.to_period("M")
first_observed_month = cohort_orders.groupby("customer_unique_id")["purchase_month"].min()
cohort_orders["cohort_month"] = cohort_orders["customer_unique_id"].map(first_observed_month)
cohort_orders = cohort_orders.loc[
    cohort_orders["cohort_month"].between(pd.Period("2017-01"), pd.Period("2017-12"))
].copy()
cohort_orders["months_since_first"] = (
    (cohort_orders["purchase_month"].dt.year - cohort_orders["cohort_month"].dt.year) * 12
    + cohort_orders["purchase_month"].dt.month - cohort_orders["cohort_month"].dt.month
)
cohort_orders = cohort_orders.loc[cohort_orders["months_since_first"].between(0, 6)]
cohort_sizes = cohort_orders.groupby("cohort_month")["customer_unique_id"].nunique()
cohort_returners = (
    cohort_orders.groupby(["cohort_month", "months_since_first"])["customer_unique_id"]
    .nunique().unstack("months_since_first", fill_value=0)
    .reindex(columns=range(7), fill_value=0)
)
cohort_repeat_rates = cohort_returners.div(cohort_sizes, axis=0)
cohort_repeat_rates.index = cohort_repeat_rates.index.astype(str)
cohort_repeat_figure = px.imshow(
    cohort_repeat_rates.loc[:, 1:6], text_auto=".1%", aspect="auto",
    color_continuous_scale="Blues", zmin=0,
    title="2017 First-Purchase Cohorts: Monthly Delivered-Order Repeat Rate",
    labels={"x": "Calendar Months After First Purchase", "y": "First-Purchase Cohort",
            "color": "Cohort Share"},
)
cohort_repeat_figure.update_coloraxes(colorbar_tickformat=".1%")
cohort_repeat_figure.update_yaxes(tickmode="array", tickvals=cohort_repeat_rates.index.tolist())
cohort_repeat_figure.update_layout(
    template="plotly_white", height=620,
    margin={"l": 100, "r": 45, "t": 85, "b": 70},
)
cohort_repeat_figure.show()
cohort_summary = pd.DataFrame({
    "first_purchase_cohort": cohort_repeat_rates.index,
    "cohort_customers": cohort_sizes.to_numpy(),
    "repeat_in_months_1_to_6": cohort_orders.loc[
        cohort_orders["months_since_first"].gt(0)
    ].groupby("cohort_month")["customer_unique_id"].nunique()
    .reindex(cohort_sizes.index, fill_value=0).to_numpy(),
})
cohort_summary["months_1_to_6_repeat_rate"] = (
    cohort_summary["repeat_in_months_1_to_6"] / cohort_summary["cohort_customers"]
)
display(cohort_summary.round(4))

,first_purchase_cohort,cohort_customers,repeat_in_months_1_to_6,months_1_to_6_repeat_rate
0,2017-01,717,12,0.0167
1,2017-02,1628,23,0.0141
2,2017-03,2503,44,0.0176
3,2017-04,2256,42,0.0186
4,2017-05,3451,69,0.0200
5,2017-06,3037,63,0.0207
6,2017-07,3752,70,0.0187
7,2017-08,4057,93,0.0229
8,2017-09,4004,87,0.0217
9,2017-10,4328,71,0.0164


**Across the twelve 2017 first-purchase cohorts, 1.41%–2.29% of consumers placed at least one delivered order in calendar months 1–6 after their first purchase month. The common six-month follow-up makes the cohort rates comparable.**


## 8. Regional Market Analysis

Customer-state views show where delivered-item GMV and customers were concentrated and how seller coverage differed across destinations.


### 8.1 Regional Demand and Customer Distribution

The offline Brazil outline is stored in source_data/brazil_boundary.geojson and derived from [Natural Earth 1:110m Admin 0](https://www.naturalearthdata.com/downloads/110m-cultural-vectors/). State points use median coordinates from the Olist geolocation file.

In [25]:
order_item_values = (
    items_df.groupby("order_id", as_index=False)
    .agg(gmv=("price", "sum"), freight_value=("freight_value", "sum"))
)

regional_orders = (
    delivered_order_details.merge(order_item_values, on="order_id", how="inner")
    .merge(
        customers_df[["customer_id", "customer_unique_id", "customer_state", "customer_city"]],
        on="customer_id",
        how="left",
    )
)

state_summary = (
    regional_orders.groupby("customer_state", as_index=False)
    .agg(
        active_customers=("customer_unique_id", "nunique"),
        delivered_orders=("order_id", "nunique"),
        gmv=("gmv", "sum"),
        freight_value=("freight_value", "sum"),
        avg_delivery_days=("delivery_days", "mean"),
        late_delivery_rate=("is_late_delivery", "mean"),
    )
)
state_summary["aov"] = state_summary["gmv"] / state_summary["delivered_orders"]
state_summary["freight_to_gmv"] = state_summary["freight_value"] / state_summary["gmv"]

state_demand_view = state_summary.nlargest(15, "gmv").sort_values("gmv")

In [26]:
# Build an offline geographic point map using local geolocation and Natural Earth data.
import json
with (DATA_DIR / "brazil_boundary.geojson").open(encoding="utf-8") as boundary_file:
    brazil_boundary = json.load(boundary_file)["geometry"]["coordinates"][0]
brazil_outline_lon, brazil_outline_lat = zip(*brazil_boundary)
valid_geo = geolocation_df.loc[
    geolocation_df["geolocation_lat"].between(-34.5, 5.5)
    & geolocation_df["geolocation_lng"].between(-74, -34)
].copy()
zip_locations = (
    valid_geo.groupby("geolocation_zip_code_prefix", as_index=False)
    .agg(latitude=("geolocation_lat", "median"), longitude=("geolocation_lng", "median"))
)
background_locations = zip_locations.sample(n=min(5000, len(zip_locations)), random_state=42)
state_locations = (
    valid_geo.groupby("geolocation_state", as_index=False)
    .agg(latitude=("geolocation_lat", "median"), longitude=("geolocation_lng", "median"))
)
state_map = state_summary.merge(state_locations, left_on="customer_state", right_on="geolocation_state", how="inner")
state_map["marker_size"] = 12 + 28 * np.sqrt(state_map["gmv"] / state_map["gmv"].max())

state_map["gmv_millions"] = state_map["gmv"] / 1_000_000
top_map_states = set(state_map.nlargest(5, "gmv")["customer_state"])
state_map["map_label"] = state_map["customer_state"].where(state_map["customer_state"].isin(top_map_states), "")
regional_demand_figure = make_subplots(
    rows=1, cols=2, specs=[[{"type": "xy"}, {"type": "map"}]],
    column_widths=[0.43, 0.57], horizontal_spacing=0.07,
    subplot_titles=("Top States by GMV", "Geographic GMV Distribution"),
)
regional_demand_figure.add_trace(go.Bar(
    x=state_demand_view["gmv"] / 1_000_000, y=state_demand_view["customer_state"],
    orientation="h", marker_color="#2563EB", text=state_demand_view["gmv"] / 1_000_000,
    texttemplate="%{text:.2f}M", textposition="outside", cliponaxis=False,
    customdata=state_demand_view[["active_customers", "delivered_orders"]].to_numpy(),
    hovertemplate="<b>%{y}</b><br>GMV: R$ %{x:.2f}M<br>Active consumers: %{customdata[0]:,.0f}<br>Delivered orders: %{customdata[1]:,.0f}<extra></extra>",
    showlegend=False,
), row=1, col=1)
regional_demand_figure.add_trace(go.Scattermap(
    lat=background_locations["latitude"], lon=background_locations["longitude"],
    mode="markers", marker={"size": 3, "color": "#CBD5E1", "opacity": 0.45},
    hoverinfo="skip", showlegend=False,
), row=1, col=2)
regional_demand_figure.add_trace(go.Scattermap(
    lat=brazil_outline_lat, lon=brazil_outline_lon, mode="lines",
    line={"color": "#334155", "width": 2}, hoverinfo="skip", showlegend=False,
), row=1, col=2)
regional_demand_figure.add_trace(go.Scattermap(
    lat=state_map["latitude"], lon=state_map["longitude"],
    mode="markers+text", text=state_map["map_label"], hovertext=state_map["customer_state"], textposition="top center",
    marker={"size": state_map["marker_size"], "color": state_map["gmv_millions"], "colorscale": "Blues", "showscale": True, "colorbar": {"title": "R$ M", "x": 1.02, "len": 0.68}},
    customdata=state_map[["gmv", "active_customers", "delivered_orders", "late_delivery_rate"]].to_numpy(),
    hovertemplate="<b>%{hovertext}</b><br>GMV: R$ %{customdata[0]:,.0f}<br>Active consumers: %{customdata[1]:,.0f}<br>Delivered orders: %{customdata[2]:,.0f}<br>Late delivery: %{customdata[3]:.1%}<extra></extra>",
    showlegend=False,
), row=1, col=2)
regional_demand_figure.update_xaxes(title_text="GMV (R$ millions)", rangemode="tozero", row=1, col=1)
regional_demand_figure.update_layout(
    title="Delivered-Order GMV by Customer State",
    map={"style": "white-bg", "center": {"lat": -15, "lon": -52}, "zoom": 3.1},
    template="plotly_white", height=680, margin={"l": 65, "r": 100, "t": 95, "b": 55},
)
regional_demand_figure.show()

**São Paulo (SP) accounts for 38.3% of delivered-item GMV. SP, RJ, MG, RS, and PR are the five largest customer states by this measure and together contribute 73.9%.**


### 8.2 State Demand, Seller Coverage, and Concentration

In [27]:
seller_state_coverage = (
    items_df[["order_id", "seller_id"]]
    .merge(delivered_order_details[["order_id", "customer_id"]], on="order_id", how="inner")
    .merge(customers_df[["customer_id", "customer_state"]], on="customer_id", how="left")
    .groupby("customer_state", as_index=False)
    .agg(active_sellers=("seller_id", "nunique"))
)
regional_opportunity = state_summary.merge(seller_state_coverage, on="customer_state", how="left")

regional_map = regional_opportunity.merge(state_locations, left_on="customer_state", right_on="geolocation_state", how="inner")
regional_map["marker_size"] = 12 + 38 * np.sqrt(regional_map["active_customers"] / regional_map["active_customers"].max())
top_opportunity_states = set(regional_map.nlargest(5, "gmv")["customer_state"])
regional_map["map_label"] = regional_map["customer_state"].where(regional_map["customer_state"].isin(top_opportunity_states), "")
delivery_rate_bounds = [regional_map["late_delivery_rate"].min(), regional_map["late_delivery_rate"].max()]
regional_opportunity_figure = make_subplots(
    rows=1, cols=2, specs=[[{"type": "xy"}, {"type": "map"}]],
    column_widths=[0.48, 0.52], horizontal_spacing=0.07,
    subplot_titles=("Demand vs. Seller Coverage", "Geographic Demand and Delivery"),
)
regional_opportunity_figure.add_trace(go.Scatter(
    x=regional_map["active_sellers"], y=regional_map["gmv"],
    mode="markers+text", text=regional_map["customer_state"].where(
        regional_map["customer_state"].eq("SP"), ""
    ), textposition="top center",
    marker={"size": regional_map["marker_size"], "color": regional_map["late_delivery_rate"],
            "colorscale": "RdYlGn_r", "cmin": delivery_rate_bounds[0], "cmax": delivery_rate_bounds[1],
            "showscale": False, "opacity": 0.8},
    customdata=regional_map[["customer_state", "active_customers", "late_delivery_rate"]].to_numpy(),
    hovertemplate="<b>%{customdata[0]}</b><br>Active sellers: %{x:,.0f}<br>GMV: R$ %{y:,.0f}<br>Active consumers: %{customdata[1]:,.0f}<br>Late delivery: %{customdata[2]:.1%}<extra></extra>",
    showlegend=False,
), row=1, col=1)
regional_opportunity_figure.add_trace(go.Scattermap(
    lat=background_locations["latitude"], lon=background_locations["longitude"],
    mode="markers", marker={"size": 3, "color": "#CBD5E1", "opacity": 0.45},
    hoverinfo="skip", showlegend=False,
), row=1, col=2)
regional_opportunity_figure.add_trace(go.Scattermap(
    lat=brazil_outline_lat, lon=brazil_outline_lon, mode="lines",
    line={"color": "#334155", "width": 2}, hoverinfo="skip", showlegend=False,
), row=1, col=2)
regional_opportunity_figure.add_trace(go.Scattermap(
    lat=regional_map["latitude"], lon=regional_map["longitude"],
    mode="markers+text", text=regional_map["map_label"], hovertext=regional_map["customer_state"], textposition="top center",
    marker={"size": regional_map["marker_size"], "color": regional_map["late_delivery_rate"],
            "colorscale": "RdYlGn_r", "cmin": delivery_rate_bounds[0], "cmax": delivery_rate_bounds[1],
            "showscale": True, "colorbar": {"title": "Late Rate", "tickformat": ".0%", "x": 1.02, "len": 0.68}},
    customdata=regional_map[["gmv", "active_customers", "active_sellers", "late_delivery_rate"]].to_numpy(),
    hovertemplate="<b>%{hovertext}</b><br>GMV: R$ %{customdata[0]:,.0f}<br>Active consumers: %{customdata[1]:,.0f}<br>Active sellers: %{customdata[2]:,.0f}<br>Late delivery: %{customdata[3]:.1%}<extra></extra>",
    showlegend=False,
), row=1, col=2)
regional_opportunity_figure.update_xaxes(title_text="Active Sellers Serving State", rangemode="tozero", row=1, col=1)
regional_opportunity_figure.update_yaxes(title_text="GMV (R$)", rangemode="tozero", row=1, col=1)
regional_opportunity_figure.update_layout(
    title="Regional Demand and Logistics Coverage",
    map={"style": "white-bg", "center": {"lat": -15, "lon": -52}, "zoom": 3.1},
    template="plotly_white", height=680, margin={"l": 75, "r": 100, "t": 95, "b": 65},
)
regional_opportunity_figure.show()

In [28]:
# Compare seller GMV concentration across customer states and map the same metric offline.
regional_seller_values = (
    items_df[["order_id", "seller_id", "price"]]
    .merge(delivered_order_details[["order_id", "customer_id"]], on="order_id", how="inner")
    .merge(customers_df[["customer_id", "customer_state"]], on="customer_id", how="inner")
    .groupby(["customer_state", "seller_id"], as_index=False)["price"].sum()
)
regional_seller_values["seller_share"] = (
    regional_seller_values["price"]
    / regional_seller_values.groupby("customer_state")["price"].transform("sum")
)
regional_state_hhi = (
    regional_seller_values.assign(squared_share=lambda frame: frame["seller_share"] ** 2)
    .groupby("customer_state", as_index=False)
    .agg(seller_hhi=("squared_share", "sum"), active_sellers=("seller_id", "nunique"))
    .merge(state_summary[["customer_state", "gmv"]], on="customer_state", how="left")
    .sort_values("seller_hhi")
)
regional_hhi_map = regional_state_hhi.merge(
    state_locations, left_on="customer_state", right_on="geolocation_state", how="inner"
)
top_hhi_states = set(regional_hhi_map.nlargest(5, "seller_hhi")["customer_state"])
regional_hhi_map["map_label"] = regional_hhi_map["customer_state"].where(
    regional_hhi_map["customer_state"].isin(top_hhi_states), ""
)
regional_hhi_map["marker_size"] = 10 + 26 * np.sqrt(
    regional_hhi_map["gmv"] / regional_hhi_map["gmv"].max()
)

regional_hhi_figure = make_subplots(
    rows=1, cols=2, specs=[[{"type": "xy"}, {"type": "map"}]],
    column_widths=[0.50, 0.50], horizontal_spacing=0.08,
    subplot_titles=("Seller HHI among Orders to State", "Geographic Seller Concentration"),
)
regional_hhi_figure.add_trace(go.Bar(
    x=regional_state_hhi["seller_hhi"], y=regional_state_hhi["customer_state"],
    orientation="h", marker_color="#2563EB",
    text=regional_state_hhi["seller_hhi"], texttemplate="%{text:.3f}", textposition="outside",
    showlegend=False,
    customdata=regional_state_hhi[["active_sellers", "gmv"]].to_numpy(),
    hovertemplate="<b>%{y}</b><br>Seller HHI: %{x:.3f}<br>Active sellers: %{customdata[0]:,.0f}<br>GMV: R$ %{customdata[1]:,.0f}<extra></extra>",
), row=1, col=1)
regional_hhi_figure.add_trace(go.Scattermap(
    lat=brazil_outline_lat, lon=brazil_outline_lon, mode="lines",
    line={"color": "#334155", "width": 2}, hoverinfo="skip", showlegend=False,
), row=1, col=2)
regional_hhi_figure.add_trace(go.Scattermap(
    lat=regional_hhi_map["latitude"], lon=regional_hhi_map["longitude"],
    mode="markers+text", text=regional_hhi_map["map_label"],
    hovertext=regional_hhi_map["customer_state"], textposition="top center",
    marker={
        "size": regional_hhi_map["marker_size"],
        "color": regional_hhi_map["seller_hhi"], "colorscale": "YlOrRd",
        "cmin": 0, "cmax": regional_hhi_map["seller_hhi"].max(),
        "opacity": 0.88, "showscale": True,
        "colorbar": {"title": "Seller HHI", "x": 1.02, "len": 0.7},
    },
    customdata=regional_hhi_map[["seller_hhi", "active_sellers", "gmv"]].to_numpy(),
    hovertemplate="<b>%{hovertext}</b><br>Seller HHI: %{customdata[0]:.3f}<br>Active sellers: %{customdata[1]:,.0f}<br>GMV: R$ %{customdata[2]:,.0f}<extra></extra>",
    showlegend=False,
), row=1, col=2)
regional_hhi_figure.update_xaxes(title_text="Seller HHI (0-1)", rangemode="tozero", row=1, col=1)
regional_hhi_figure.update_yaxes(title_text="Customer State", row=1, col=1)
regional_hhi_figure.update_layout(
    title="Concentration of Sellers Serving Each Customer State",
    map={"style": "white-bg", "center": {"lat": -15, "lon": -52}, "zoom": 3.1},
    template="plotly_white", height=790,
    margin={"l": 75, "r": 105, "t": 95, "b": 60},
)
regional_hhi_figure.show()


**RJ generated approximately R$1.76 million in delivered-item GMV, making it one of Olist's largest customer-state markets. Its observed late-delivery rate was 13.5%, compared with 5.9% in SP. This combination of substantial order activity and a higher late-delivery rate identifies RJ as a useful state for examining delivery performance alongside customer reviews.**

**Seller HHI measures how concentrated GMV from a state's buyers is among the sellers serving them. RR and AP have the highest observed state-level HHI values, approximately 0.059 and 0.047, while SP is approximately 0.004. The higher values indicate more concentrated observed sales in those states; they do not establish that the sellers are local, that local supply is insufficient, or that adding sellers would increase demand.**

Active sellers are sellers on delivered orders shipped to customers in each state. Points on the maps mark representative customer-state locations.


## 9. Delivery and Logistics Analysis

This section describes observed delivery time, estimated-date performance, and the distribution of recorded order statuses.


### 9.1 Delivery Time, Timeliness, and Delays

In [29]:
# Find the first delivery day whose mean review score falls below the overall mean.
delivery_performance = delivered_order_details.loc[
    delivered_order_details["delivery_days"].notna()
    & delivered_order_details["delivery_days"].ge(0)
    & delivered_order_details["delivery_delay_days"].notna()
].copy()
on_time_rate = 1 - delivery_performance["is_late_delivery"].mean()
latest_delivery_reviews = (
    reviews_df.dropna(subset=["review_score"])
    .sort_values("review_answer_timestamp", na_position="first")
    .drop_duplicates(subset="order_id", keep="last")
)
delivery_review_days = delivery_performance[["order_id", "delivery_days"]].merge(
    latest_delivery_reviews[["order_id", "review_score"]], on="order_id", how="inner"
)
overall_delivered_review_mean = delivery_review_days["review_score"].mean()
delivery_time_limit = delivery_review_days["delivery_days"].quantile(0.99)
delivery_review_days = delivery_review_days.loc[delivery_review_days["delivery_days"].le(delivery_time_limit)].copy()
delivery_review_days["delivery_day"] = np.floor(delivery_review_days["delivery_days"]).astype(int)
delivery_review_days["low_review"] = delivery_review_days["review_score"].le(2)
delivery_review_days["high_review"] = delivery_review_days["review_score"].ge(4)
delivery_day_summary = (
    delivery_review_days.groupby("delivery_day", as_index=False)
    .agg(
        reviewed_orders=("order_id", "nunique"),
        low_review_rate=("low_review", "mean"),
        high_review_rate=("high_review", "mean"),
        average_review_score=("review_score", "mean"),
    )
)
delivery_day_summary = delivery_day_summary.loc[delivery_day_summary["reviewed_orders"].ge(30)]
below_average_days = delivery_day_summary.loc[
    delivery_day_summary["average_review_score"].lt(overall_delivered_review_mean)
]
review_below_average_day = int(below_average_days["delivery_day"].iloc[0]) if not below_average_days.empty else None

# Select the best three-parameter distribution by in-sample R-squared across all displayed day bins.
from scipy.optimize import least_squares
from scipy.stats import gamma, lognorm, weibull_min
plotted_delivery_days = delivery_review_days.loc[
    delivery_review_days["delivery_day"].isin(delivery_day_summary["delivery_day"]), "delivery_day"
]
delivery_day_skewness = plotted_delivery_days.skew()
delivery_day_excess_kurtosis = plotted_delivery_days.kurt()
delivery_midpoints = plotted_delivery_days.to_numpy(dtype=float) + 0.5
delivery_mean = delivery_midpoints.mean()
delivery_variance = delivery_midpoints.var()
moment_shape = delivery_mean ** 2 / delivery_variance
moment_scale = delivery_variance / delivery_mean
fit_days = delivery_day_summary["delivery_day"].to_numpy(dtype=float)
observed_day_counts = delivery_day_summary["reviewed_orders"].to_numpy(dtype=float)
total_squared_deviation = np.sum((observed_day_counts - observed_day_counts.mean()) ** 2)

def distribution_day_counts(parameters, distribution):
    amplitude, shape, scale = parameters
    day_probability = (
        distribution.cdf(fit_days + 1, shape, scale=scale)
        - distribution.cdf(fit_days, shape, scale=scale)
    )
    return amplitude * day_probability

candidate_distributions = {
    "Gamma": (gamma, moment_shape, moment_scale),
    "Lognormal": (lognorm, np.sqrt(np.log1p(delivery_variance / delivery_mean ** 2)),
                  delivery_mean / np.sqrt(1 + delivery_variance / delivery_mean ** 2)),
    "Weibull": (weibull_min, 1.5, delivery_mean),
}
distribution_fits = []
for distribution_name, (distribution, initial_shape, initial_scale) in candidate_distributions.items():
    for scale_factor in (0.75, 1.0, 1.25):
        initial_parameters = [observed_day_counts.sum(), initial_shape, initial_scale * scale_factor]
        result = least_squares(
            lambda parameters: distribution_day_counts(parameters, distribution) - observed_day_counts,
            x0=initial_parameters, bounds=([1, 0.05, 0.1], [np.inf, 50, 200]),
            max_nfev=10000,
        )
        fitted_counts = distribution_day_counts(result.x, distribution)
        residual_sum_of_squares = np.sum((observed_day_counts - fitted_counts) ** 2)
        distribution_fits.append((1 - residual_sum_of_squares / total_squared_deviation,
                                  distribution_name, result.x, fitted_counts))
fit_r_squared, fit_name, fit_parameters, fitted_day_counts = max(
    distribution_fits, key=lambda fitted: fitted[0]
)
fit_amplitude, fit_shape, fit_scale = fit_parameters

delivery_review_figure = make_subplots(
    rows=1, cols=2, column_widths=[0.54, 0.46], horizontal_spacing=0.11,
    subplot_titles=("Mean Review vs. Overall Average", "Review Volume and Mean Score"),
)
delivery_review_figure.add_trace(go.Scatter(
    x=delivery_day_summary["delivery_day"], y=delivery_day_summary["average_review_score"],
    mode="lines+markers", name="Daily mean review",
    line={"color": "#2563EB", "width": 2.5}, marker={"size": 5},
), row=1, col=1)
delivery_review_figure.add_hline(
    y=overall_delivered_review_mean, line_color="#64748B", line_dash="dash",
    annotation_text=f"Overall mean: {overall_delivered_review_mean:.2f}", row=1, col=1,
)
delivery_review_figure.add_trace(go.Bar(
    x=delivery_day_summary["delivery_day"], y=delivery_day_summary["reviewed_orders"],
    marker={
        "color": delivery_day_summary["average_review_score"], "colorscale": "RdYlGn",
        "cmin": 1, "cmax": 5, "showscale": True,
        "colorbar": {"title": "Mean Review", "x": 1.02, "len": 0.75},
    },
    customdata=delivery_day_summary[["average_review_score", "low_review_rate", "high_review_rate"]].to_numpy(),
    hovertemplate="Delivery: %{x} days<br>Reviewed orders: %{y:,.0f}<br>Mean review: %{customdata[0]:.2f} / 5<br>Low score: %{customdata[1]:.1%}<br>High score: %{customdata[2]:.1%}<extra></extra>",
    showlegend=False,
), row=1, col=2)
delivery_review_figure.add_trace(go.Scatter(
    x=delivery_day_summary["delivery_day"], y=fitted_day_counts,
    mode="lines", name=f"{fit_name} fit (R²={fit_r_squared:.3f})",
    line={"color": "#1D4ED8", "width": 3},
    hovertemplate="Delivery: %{x} days<br>Fitted reviewed orders: %{y:,.0f}<extra></extra>",
), row=1, col=2)
if review_below_average_day is not None:
    delivery_review_figure.add_vline(
        x=review_below_average_day, line_color="#7C3AED", line_dash="dash", line_width=2,
        annotation_text=f"First below mean: {review_below_average_day} days", row=1, col=1,
    )
    delivery_review_figure.add_vline(
        x=review_below_average_day, line_color="#7C3AED", line_dash="dash", line_width=2,
        row=1, col=2,
    )
delivery_review_figure.update_xaxes(title_text="Purchase-to-Delivery Days")
delivery_review_figure.update_yaxes(title_text="Mean Review Score (1-5)", range=[1, 5], row=1, col=1)
delivery_review_figure.update_yaxes(title_text="Reviewed Orders", row=1, col=2)
delivery_review_figure.update_layout(
    title="Delivery Time and Review Outcomes",
    template="plotly_white", height=600,
    legend={"orientation": "h", "y": -0.12, "x": 0.02},
    margin={"l": 70, "r": 100, "t": 100, "b": 105},
)
delivery_review_figure.show()
display(pd.DataFrame([{
    "Fitted Distribution": fit_name,
    "In-Sample R Squared": fit_r_squared,
    "Amplitude": fit_amplitude,
    "Shape": fit_shape,
    "Scale": fit_scale,
    "Delivery-Day Skewness": delivery_day_skewness,
    "Delivery-Day Excess Kurtosis": delivery_day_excess_kurtosis,
    "Minimum Reviews per Day": 30,
}]).round(3))
print(
    f"First delivery day with mean review below the overall mean: {review_below_average_day} days"
    if review_below_average_day is not None else "No reviewed delivery-day bin falls below the overall mean."
)

,Fitted Distribution,In-Sample R Squared,Amplitude,Shape,Scale,Delivery-Day Skewness,Delivery-Day Excess Kurtosis,Minimum Reviews per Day
0,Gamma,0.988,92321.106,2.932,3.858,1.388,2.259,30


First delivery day with mean review below the overall mean: 16 days


**Among reviewed delivered orders, mean score generally declines as purchase-to-delivery time increases. Day 16 is the first displayed day-bin below the overall mean under the chart's 30-review minimum and 99th-percentile duration cutoff.**

The review-count panel fits Gamma, lognormal, and Weibull distributions to the displayed day bins and shows the highest in-sample R² fit. The table above reports the fitted parameters, skewness, and excess kurtosis.


### 9.2 Order Status and Delivery Exceptions

In [30]:
order_status_summary = (
    orders_df.groupby("order_status", as_index=False)
    .agg(orders=("order_id", "nunique"))
    .sort_values("orders", ascending=False)
)
order_status_summary["order_share"] = order_status_summary["orders"] / order_status_summary["orders"].sum()
delivered_count = int(order_status_summary.loc[order_status_summary["order_status"].eq("delivered"), "orders"].sum())
other_count = int(order_status_summary["orders"].sum() - delivered_count)
other_share = other_count / (delivered_count + other_count)
other_slice_angle = 360 * other_share
other_statuses = order_status_summary.loc[order_status_summary["order_status"].ne("delivered")].copy()
status_colors = {"delivered": "#2563EB", "shipped": "#0F766E", "canceled": "#EA580C", "unavailable": "#DC2626", "invoiced": "#7C3AED", "processing": "#F59E0B", "approved": "#64748B", "created": "#94A3B8"}
order_status_figure = go.Figure()
order_status_figure.add_trace(go.Pie(
    labels=["Other statuses", "Delivered"], values=[other_count, delivered_count],
    hole=0.62, domain={"x": [0.00, 0.40], "y": [0.12, 0.90]},
    marker={"colors": ["#EA580C", status_colors["delivered"]]},
    textinfo="percent", textposition="inside", sort=False, direction="clockwise", rotation=90 - other_slice_angle / 2, showlegend=False,
    hovertemplate="%{label}: %{value:,.0f} orders (%{percent})<extra></extra>",
))
order_status_figure.add_trace(go.Pie(
    labels=other_statuses["order_status"].str.title(), values=other_statuses["orders"],
    hole=0.62, domain={"x": [0.60, 1.00], "y": [0.12, 0.90]},
    marker={"colors": [status_colors.get(status, "#94A3B8") for status in other_statuses["order_status"]]},
    textinfo="label+percent", textposition="outside", sort=False, showlegend=False,
    hovertemplate="%{label}: %{value:,.0f} orders (%{percent} of non-delivered)<extra></extra>",
))
order_status_figure.update_layout(
    title="Order Outcomes and Non-Delivered Status Breakdown", template="plotly_white", height=550,
    annotations=[
        {"x": 0.20, "y": 0.51, "xref": "paper", "yref": "paper", "text": f"All orders<br>{delivered_count + other_count:,}", "showarrow": False, "font": {"size": 16}},
        {"x": 0.80, "y": 0.51, "xref": "paper", "yref": "paper", "text": f"Non-delivered<br>{other_count:,}", "showarrow": False, "font": {"size": 16}},
        {"x": 0.50, "y": 0.62, "xref": "paper", "yref": "paper", "text": f"Other statuses<br>{other_share:.1%} of all orders", "showarrow": False, "font": {"size": 13, "color": "#EA580C"}},
        {"x": 0.60, "y": 0.51, "ax": -20, "ay": 0, "xref": "paper", "yref": "paper", "text": " ", "showarrow": True, "arrowhead": 3, "arrowsize": 1.2, "arrowwidth": 2.5, "arrowcolor": "#EA580C"},
    ],
    shapes=[{"type": "line", "xref": "paper", "yref": "paper", "x0": 0.40, "x1": 0.58, "y0": 0.51, "y1": 0.51, "line": {"color": "#EA580C", "width": 2.5}}],
    margin={"l": 50, "r": 70, "t": 90, "b": 40},
)
order_status_figure.show()

**Of 99,441 recorded orders, 96,478 (97.0%) have delivered status. The remaining 2,963 include 1,107 marked shipped, 625 canceled, and 609 unavailable, along with smaller status groups. The right-hand chart describes the composition of these non-delivered statuses; it should not be read as a single delivery-failure rate.**


The left ring shows delivered versus all other orders. The connected right ring breaks down only the non-delivered group, so its percentages use non-delivered orders as the denominator.

## 10. Customer Experience and Review Analysis

Review scores are summarized across scored orders and compared by customer state. The delivery-time relationship appears in Section 9.1.


### 10.1 Review Score Distribution and Satisfaction Levels

In [31]:
order_reviews = (
    reviews_df.dropna(subset=["review_score"])
    .sort_values("review_answer_timestamp", na_position="first")
    .drop_duplicates(subset="order_id", keep="last")
)
review_distribution = (
    order_reviews.groupby("review_score", as_index=False)
    .agg(reviews=("order_id", "nunique"))
    .sort_values("review_score")
)
review_distribution["review_share"] = review_distribution["reviews"] / review_distribution["reviews"].sum()
average_review_score = order_reviews["review_score"].mean()

review_distribution_figure = px.bar(
    review_distribution,
    x="review_score",
    y="reviews",
    color="review_score",
    color_continuous_scale=["#D73027", "#FC8D59", "#FEE08B", "#91CF60", "#1A9850"],
    text="review_share",
    title=f"Review Score Distribution | Average Score: {average_review_score:.2f}",
    labels={"review_score": "Review Score", "reviews": "Reviewed Orders"},
)
review_distribution_figure.update_traces(texttemplate="%{text:.1%}", textposition="outside")
review_distribution_figure.update_layout(template="plotly_white", height=500, coloraxis_showscale=False)
review_distribution_figure.show()

**Five-star reviews account for 57.8% of scored orders, followed by four-star (19.3%), one-star (11.5%), three-star (8.2%), and two-star (3.2%). The combined low-score share is 14.7%, the high-score share is 77.1%, and the mean score is 4.09 out of five.**


### 10.2 Average Review Score by Customer State

Each delivered order contributes at most one review. The reference mean here is 4.16 for scored delivered orders; Section 10.1's 4.09 includes all scored orders regardless of status. All customer states are shown. Colors represent the difference from the delivered-order mean in units of the review-count-weighted standard deviation of state means: green is above the mean, red is below it, and gray is near it. Bubble size represents reviewed-order count.

In [32]:
# Rank customer states by mean review score and show their locations offline.
state_review_summary = (
    order_reviews[["order_id", "review_score"]]
    .merge(delivered_order_details[["order_id", "customer_id"]], on="order_id", how="inner")
    .merge(customers_df[["customer_id", "customer_state"]], on="customer_id", how="left")
    .dropna(subset=["customer_state"])
    .groupby("customer_state", as_index=False)
    .agg(reviewed_orders=("order_id", "nunique"), average_review_score=("review_score", "mean"))
)
olist_delivered_review_mean = np.average(
    state_review_summary["average_review_score"], weights=state_review_summary["reviewed_orders"]
)
state_mean_sd = np.sqrt(np.average(
    (state_review_summary["average_review_score"] - olist_delivered_review_mean) ** 2,
    weights=state_review_summary["reviewed_orders"],
))
state_review_summary["score_z"] = (
    (state_review_summary["average_review_score"] - olist_delivered_review_mean) / state_mean_sd
)
state_review_rank = state_review_summary.sort_values("average_review_score")
state_review_map = state_review_summary.merge(
    state_locations, left_on="customer_state", right_on="geolocation_state", how="inner"
)
state_review_map["marker_size"] = 10 + 30 * np.sqrt(
    state_review_map["reviewed_orders"] / state_review_map["reviewed_orders"].max()
)
top_review_states = set(state_review_map.nlargest(5, "reviewed_orders")["customer_state"])
state_review_map["map_label"] = state_review_map["customer_state"].where(
    state_review_map["customer_state"].isin(top_review_states), ""
)
state_review_figure = make_subplots(
    rows=1, cols=2, specs=[[{"type": "xy"}, {"type": "map"}]],
    column_widths=[0.46, 0.54], horizontal_spacing=0.07,
    subplot_titles=("Mean Review Score Ranking", "Geographic Review Scores"),
)
state_review_figure.add_trace(go.Bar(
    x=state_review_rank["average_review_score"], y=state_review_rank["customer_state"],
    orientation="h", marker={"color": state_review_rank["score_z"],
        "colorscale": [[0, "#D73027"], [0.5, "#E5E7EB"], [1, "#1A9850"]],
        "cmin": -2, "cmax": 2},
    customdata=state_review_rank[["reviewed_orders", "score_z"]].to_numpy(),
    hovertemplate="<b>%{y}</b><br>Mean review: %{x:.2f} / 5<br>Difference: %{customdata[1]:+.2f} SD<br>Reviewed orders: %{customdata[0]:,.0f}<extra></extra>",
    showlegend=False,
), row=1, col=1)
state_review_figure.add_vline(
    x=olist_delivered_review_mean, line_color="#475569", line_dash="dash",
    annotation_text=f"Olist delivered mean {olist_delivered_review_mean:.2f}", row=1, col=1,
)
state_review_figure.add_trace(go.Scattermap(
    lat=brazil_outline_lat, lon=brazil_outline_lon, mode="lines",
    line={"color": "#334155", "width": 2}, hoverinfo="skip", showlegend=False,
), row=1, col=2)
state_review_figure.add_trace(go.Scattermap(
    lat=state_review_map["latitude"], lon=state_review_map["longitude"],
    mode="markers+text", text=state_review_map["map_label"],
    hovertext=state_review_map["customer_state"], textposition="top center",
    marker={
        "size": state_review_map["marker_size"],
        "color": state_review_map["score_z"],
        "colorscale": [[0, "#D73027"], [0.5, "#E5E7EB"], [1, "#1A9850"]],
        "cmin": -2, "cmax": 2,
        "showscale": True, "colorbar": {"title": "Difference", "x": 1.02, "len": 0.7,
            "tickvals": [-2, -1, 0, 1, 2],
            "ticktext": ["-2 SD", "-1 SD", "Mean", "+1 SD", "+2 SD"]},
    },
    customdata=state_review_map[["average_review_score", "reviewed_orders", "score_z"]].to_numpy(),
    hovertemplate="<b>%{hovertext}</b><br>Mean review: %{customdata[0]:.2f} / 5<br>Difference: %{customdata[2]:+.2f} SD<br>Reviewed orders: %{customdata[1]:,.0f}<extra></extra>",
    showlegend=False,
), row=1, col=2)
state_review_figure.update_xaxes(title_text="Mean Review Score (1-5)", range=[0, 5], row=1, col=1)
state_review_figure.update_layout(
    title=f"Customer-State Review Scores | Olist Delivered Mean {olist_delivered_review_mean:.2f}, 1 SD {state_mean_sd:.3f}",
    map={"style": "white-bg", "center": {"lat": -15, "lon": -52}, "zoom": 3.1},
    template="plotly_white", height=760, margin={"l": 75, "r": 105, "t": 95, "b": 55},
)
state_review_figure.show()

**SP, PR, AP, and AM have observed mean review scores above the reviewed-order average; MG and RS are slightly above it. RJ averages 3.97 across 12,211 reviewed delivered orders, below the Olist delivered-order mean of 4.16. Its combination of substantial order volume and lower mean score makes RJ a useful state-level priority for investigating service quality.**


## 11. Key Findings

1. **Marketplace activity.** Delivered items generated R$13.22 million in item-price GMV across 96,478 orders, or R$137.04 per delivered order. The observed delivered-order population includes 93,358 customers and 2,970 sellers. Monthly GMV and delivered-order counts generally increased across the complete months of 2017–August 2018.
2. **Product concentration.** Seventeen of 73 named categories (23.3%) contributed approximately 80% of classified delivered-item GMV. Each of the five leading displayed categories generated more GMV in January–August 2018 than in the same months of 2017; comparable-window increases ranged from 109.2% to 242.0%.
3. **Customer behavior.** Credit cards carried the most payment value. Among customers with delivered orders, 3.00% placed at least two such orders in the available records. In equal six-month follow-up windows, the twelve 2017 first-purchase cohorts recorded subsequent-month repeat rates of 1.41%–2.29%. The highest-spending 5% contributed 26.8% of observed customer spending.
4. **Geography and delivery.** SP alone contributed 38.3% of delivered-item GMV; SP, RJ, MG, RS, and PR together contributed 73.9%. Buyer-charged freight totaled 16.6% of item-price GMV, and 91.9% of eligible delivered orders arrived by their estimated date.
5. **Customer experience.** The mean score across all scored orders was 4.09 out of five. Among reviewed delivered orders, mean score generally declined as purchase-to-delivery time increased; day 16 was the first displayed delivery-day group below its overall mean under the stated bin rules.


## 12. Business Recommendations

The following operating priorities pair each observed pattern with a measurable next step.

1. **Investigate first-to-second purchase conversion.** Track equal-maturity cohorts by first purchase month, category, and acquisition channel. Test a defined post-purchase follow-up against a control group, measuring six-month repeat purchase, cost per retained customer, and incremental contribution. The low observed repeat rates make this a focused question for a retention experiment.
2. **Monitor delivery and ratings together.** Review on-time rate, delivery-day distribution, mean score, and low-score share by customer state and product category. Prioritize combinations with enough orders to compare reliably, then test a logistics change and assess both delivery performance and review outcomes.
3. **Review the leading category portfolio.** Monitor the categories accounting for most delivered-item GMV, using the same calendar months when comparing years. Combine GMV with seller concentration, charged freight, and review results before changing category emphasis.
4. **Use state-level demand with service capacity.** SP, RJ, MG, RS, and PR account for most observed GMV. Compare their order volume, seller coverage, freight charge, delivery time, and ratings when planning local service improvements. Assess population and market-access differences before interpreting a smaller state total as untapped demand.
5. **Preserve payment-method coverage.** Credit cards and boleto account for the largest observed payment values. Track payment success and checkout completion by method if those event-level data become available; the current payment file records completed payment entries rather than checkout attempts.


## 13. Analysis Limitations

The findings describe Olist transactions in the supplied 2016–2018 extract, not all Brazilian e-commerce activity. The data cannot measure national market share, brand position, household purchasing power, or seller profit. Item-price GMV excludes buyer-charged freight, and the first and last calendar years are partial.

Orders, items, payments, reviews, and geolocations have different grains. The analysis sets its grain before aggregation; review results use scored orders, and customer-state maps use representative points rather than state polygons. State totals are unadjusted for population.

Customers enter the extract at different times. The fixed-window cohort analysis therefore uses 2017 first-purchase groups with six subsequent calendar months of observation. Delivery–review comparisons are observational. The day-16 review crossing depends on the displayed bin threshold and duration cutoff. The three-month forecast uses 20 months of history and reports confidence intervals for fitted mean values without an out-of-time test.


## 14. Conclusion

This notebook connects marketplace scale with category mix, customer behavior, regional concentration, delivery performance, and reviews in one reproducible analysis. Olist's observed GMV and delivered-order volume rose over the comparable monthly series, while sales and customer spending were concentrated among a minority of categories, states, and buyers. Repeat purchasing was uncommon in the available records, and longer observed delivery times coincided with lower review scores.

The practical next step is to test retention and service improvements with defined populations and observation windows, then monitor whether customer behavior and ratings change. The notebook supplies the historical baseline and metric definitions for those tests.


## 15. Appendix

### 15.1 Metric Definitions

| Metric | Definition and eligible population |
|---|---|
| Delivered-item GMV | Sum of item `price` on orders with `delivered` status, in BRL; excludes `freight_value`. |
| Delivered orders and average order value | Distinct delivered `order_id` count; average order value equals delivered-item GMV divided by that count. |
| Active customer and seller | Distinct `customer_unique_id` and `seller_id` attached to delivered orders or their items. |
| Category GMV share | Category delivered-item GMV divided by all delivered-item GMV, including the Unknown bucket. |
| Pareto crossing | The smallest number of named categories reaching at least 80% of delivered-item GMV among products with a known category. |
| Equal-window category GMV change | (January–August 2018 category GMV / January–August 2017 category GMV) − 1. |
| Seller HHI within a category | Sum of squared seller shares of delivered-item GMV within that category. |
| Seller HHI by customer state | Sum of squared seller shares of delivered-item GMV from orders shipped to customers in that state. |
| Charged freight / item-price GMV | Sum of item `freight_value` divided by sum of item `price` on eligible delivered items. |
| 2017 active-month and active-year spend | Mean payment value per distinct observed customer-month and mean total 2017 payment value per distinct observed customer, respectively. |
| Observed repeat customer | Customer with at least two distinct delivered orders in the stated observation window. |
| Subsequent-month cohort repeat rate | Share of a 2017 first-purchase cohort with at least one delivered order in calendar months 1–6 after its first observed purchase month. |
| On-time delivery rate | Share of eligible delivered orders with actual customer-delivery time no later than the estimated delivery date. |
| Low- and high-score review shares | Shares of scored orders with scores 1–2 and 4–5, respectively; at most the latest answered review is retained per order. |
| State review standard score | (State mean review score − dataset-wide reviewed-delivered-order mean) / review-count-weighted standard deviation of state means. |

### 15.2 Sources, Joins, and Reproducibility

The notebook reads nine Olist CSV files from `source_data/` and the offline Brazil outline in `source_data/brazil_boundary.geojson`. Orders join to customers on `customer_id`; items, payments, and reviews connect on `order_id`; items connect to products on `product_id` and sellers on `seller_id`. Portuguese product categories use `product_category_name_translation.csv`.

Section 4 reports file-level missingness, validates transaction keys, and removes exact duplicate geolocation rows. Section 7.4 gives 2017 first-purchase cohorts the same six subsequent calendar months of follow-up. Section 9.1 displays nonnegative delivery durations through the 99th percentile and requires at least 30 scored reviews per displayed day-bin. Run the notebook from its first cell after replacing or updating any source CSV.
